# LDC LPI Species Dictionary — Reproducible End-to-End Build

This notebook builds the national LDC LPI interpretation dictionary in one direction:

**observed codes → protocol resolution → USDA taxonomy → cache audit/recovery → ecological traits → two-level MOSAIC functional groups → universal hit interpretation → hard QA → one final master write**

## Design rules

- `observed_code` is immutable provenance.
- True source blanks become `__NO_CANOPY__`; literal USDA symbol `NONE` remains a plant taxon.
- Corrected/synonymized species codes are stored separately in `USDA_accepted_symbol`; the observed field code is never overwritten.
- USDA taxonomy and USDA ecological traits are separate data sources.
- `USDA_PLANTS_ecological_attributes.csv` is the single evolving ecological-trait cache.
- The notebook audits that cache against the **actual accepted USDA symbols required by this dictionary** and repairs only absent/unverified incomplete profiles with the validated root-level USDA `PlantProfile` parser.
- USDA source trait strings are retained, including pipe-delimited values.
- `USDA_native_status_L48` is standardized to `N` / `I`; `USDA_native_type_L48` stores `Native` / `Introduced`.
- `MOSAIC_FG_structural` describes growth form + life history without requiring nativity.
- `MOSAIC_FG` adds nativity where available and otherwise falls back to the structural label.
- Explicit protocol/project functional information has precedence over USDA-derived fallback.
- Every dictionary row receives a non-null `hit_interpretation`, including unresolved oddball codes, which fall back explicitly to an unknown class rather than disappearing.
- The final master is written **once**, only after all hard QA checks pass.

The result is intended to be a complete lookup table for all ~16 million LPI hits: every source code is retained, every correction is explicit, and every hit remains interpretable even when the best available interpretation is `UnknownPlant` or another broad/unknown class.


In [91]:
# ============================================================================
# 1. IMPORTS AND AUTHORITATIVE PATHS
# ============================================================================

from pathlib import Path
from datetime import datetime, timezone
import re
import time
import shutil

import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

LPI_CODE_FILE = BASE_DIR / "LDC_LPI_observed_code_unique.csv"
USDA_PLANTS_SOURCE = BASE_DIR / "plantlst.txt"

OUTPUT_DIR = BASE_DIR / "species_dictionary_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Single evolving USDA ecological-trait cache.
USDA_ATTRIBUTE_CACHE = (
    OUTPUT_DIR / "USDA_PLANTS_ecological_attributes.csv"
)

USDA_RECOVERY_FAILURE_FILE = (
    OUTPUT_DIR / "USDA_PLANTS_ecological_attributes_recovery_failures.csv"
)

# Final production artifact.
FINAL_MASTER_FILE = (
    OUTPUT_DIR / "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

# USDA service configuration.
USDA_SERVICE_BASE = "https://plantsservices.sc.egov.usda.gov/api"
REQUEST_TIMEOUT = 30
REQUEST_DELAY_SECONDS = 0.25
MAX_RETRIES = 3
SAVE_EVERY = 25

# Required source files. The ecological-trait cache is allowed to be absent:
# if absent, this notebook will create it from the USDA profiles required by
# the current accepted-symbol universe.
for p in [
    LPI_CODE_FILE,
    USDA_PLANTS_SOURCE,
]:
    assert p.exists(), f"Missing required input: {p}"

print("Observed-code universe:", LPI_CODE_FILE)
print("USDA taxonomy backbone:", USDA_PLANTS_SOURCE)
print("USDA ecological-trait cache:", USDA_ATTRIBUTE_CACHE)
print("Final master output:", FINAL_MASTER_FILE)


Observed-code universe: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_observed_code_unique.csv
USDA taxonomy backbone: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\plantlst.txt
USDA ecological-trait cache: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\USDA_PLANTS_ecological_attributes.csv
Final master output: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv


## 2. Load and canonicalize observed LPI code universe

In [92]:
# =========================================================================
# 2. LOAD AND CANONICALIZE OBSERVED LPI CODE UNIVERSE
# =========================================================================

codes = pd.read_csv(
    LPI_CODE_FILE,
    dtype={"code": "string"},
    low_memory=False
)

print("Input rows:", f"{len(codes):,}")


# -------------------------------------------------------------------------
# Preserve the original source code
# -------------------------------------------------------------------------

codes["observed_code_raw"] = codes["code"]


# -------------------------------------------------------------------------
# Identify true source blanks BEFORE canonicalization
# -------------------------------------------------------------------------

codes["source_code_was_blank"] = (
    codes["observed_code_raw"].isna()
    | codes["observed_code_raw"].str.strip().eq("")
)


# -------------------------------------------------------------------------
# Canonical observed code
#
# IMPORTANT:
# "__NO_CANOPY__" = blank source TopCanopy observation
# "NONE"          = legitimate literal USDA plant symbol
# -------------------------------------------------------------------------

codes["observed_code"] = (
    codes["observed_code_raw"]
    .astype("string")
    .str.strip()
)

codes.loc[
    codes["source_code_was_blank"],
    "observed_code"
] = "__NO_CANOPY__"


# -------------------------------------------------------------------------
# Basic QA
# -------------------------------------------------------------------------

print(
    "Canonical observed codes:",
    f"{codes['observed_code'].nunique(dropna=False):,}"
)

print(
    "Source-blank rows:",
    f"{codes['source_code_was_blank'].sum():,}"
)

print(
    "Literal NONE rows:",
    f"{codes['observed_code'].eq('NONE').sum():,}"
)

display(
    codes[
        codes["observed_code"].isin(
            ["__NO_CANOPY__", "NONE"]
        )
    ][
        [
            "observed_code",
            "observed_code_raw",
            "source_code_was_blank",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
)

Input rows: 10,960
Canonical observed codes: 10,960
Source-blank rows: 1
Literal NONE rows: 1


,observed_code,observed_code_raw,source_code_was_blank,n_records,n_top,n_lower,n_soil_surface
15,__NO_CANOPY__,<NA>,True,442612,442612,0,0
2527,NONE,NONE,False,398,394,4,0


## 3. Apply conservative protocol-aware rules

In [93]:
# =========================================================================
# 3. CONSERVATIVE PROTOCOL-AWARE RULES
# =========================================================================

protocol_rules = pd.DataFrame(
    [
        ("__NO_CANOPY__", "No top-canopy contact", "NoCanopy"),

        ("S",  "Soil", "NonPlant"),
        ("R",  "Rock", "NonPlant"),
        ("BR", "Bedrock", "NonPlant"),
        ("GR", "Gravel", "NonPlant"),
        ("CB", "Cobble", "NonPlant"),
        ("ST", "Stone", "NonPlant"),
        ("BY", "Boulder", "NonPlant"),

        ("HL", "Herbaceous litter", "NonPlant"),
        ("WL", "Woody litter", "NonPlant"),
        ("NL", "Other litter", "NonPlant"),

        ("L",  "Lichen", "NonPlant"),
        ("LC", "Lichen / biotic crust", "NonPlant"),
        ("VL", "Vagrant lichen", "NonPlant"),
        ("M",  "Moss", "NonPlant"),

        ("EL", "Embedded litter", "NonPlant"),
        ("D",  "Duff", "NonPlant"),
        ("WA", "Water", "NonPlant"),

        # Source/protocol-dependent meanings
        ("W",  "Context-dependent W", "ContextDependent"),
        ("AG", "Context-dependent AG", "ContextDependent"),
        ("PC", "Context-dependent PC", "ContextDependent"),

        # Not safely interpreted yet
        ("O",  "Unresolved protocol code O", "ProtocolUnresolved"),
        ("PT", "Unresolved protocol code PT", "ProtocolUnresolved"),
    ],
    columns=[
        "observed_code",
        "protocol_label",
        "protocol_code_class",
    ]
)

print("Protocol rules:", len(protocol_rules))

display(protocol_rules)

Protocol rules: 23


,observed_code,protocol_label,protocol_code_class
0,__NO_CANOPY__,No top-canopy contact,NoCanopy
1,S,Soil,NonPlant
2,R,Rock,NonPlant
3,BR,Bedrock,NonPlant
4,GR,Gravel,NonPlant
5,CB,Cobble,NonPlant
6,ST,Stone,NonPlant
7,BY,Boulder,NonPlant
8,HL,Herbaceous litter,NonPlant
9,WL,Woody litter,NonPlant


## 4. Construct fresh dictionary

In [94]:
# =========================================================================
# 4. CONSTRUCT FRESH MASTER DICTIONARY
# =========================================================================

dictionary = codes.copy()

dictionary = dictionary.merge(
    protocol_rules,
    on="observed_code",
    how="left",
    validate="many_to_one",
)

dictionary["code_class"] = (
    dictionary["protocol_code_class"]
    .astype("string")
)

dictionary["resolved_label"] = (
    dictionary["protocol_label"]
    .astype("string")
)


# -------------------------------------------------------------------------
# QA
# -------------------------------------------------------------------------

print(
    "Dictionary rows:",
    f"{len(dictionary):,}"
)

print(
    "Unique observed codes:",
    f"{dictionary['observed_code'].nunique(dropna=False):,}"
)

print("\nCurrent classification:")

display(
    dictionary["code_class"]
    .value_counts(dropna=False)
    .rename_axis("code_class")
    .reset_index(name="n_codes")
)

Dictionary rows: 10,960
Unique observed codes: 10,960

Current classification:


,code_class,n_codes
0,<NA>,10937
1,NonPlant,17
2,ContextDependent,3
3,ProtocolUnresolved,2
4,NoCanopy,1


## 5. Load USDA PLANTS taxonomy backbone

In [95]:
# =========================================================================
# 5. LOAD USDA PLANTS TAXONOMIC REFERENCE — DEFENSIVE PARSER
# =========================================================================

print("USDA source:")
print(USDA_PLANTS_SOURCE)


# -------------------------------------------------------------------------
# Inspect raw header first
# -------------------------------------------------------------------------

with open(
    USDA_PLANTS_SOURCE,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:
    raw_header = f.readline()

print("\nRaw file header:")
print(repr(raw_header))


# -------------------------------------------------------------------------
# Expected USDA PLANTS fields
# -------------------------------------------------------------------------

required_usda_cols = [
    "Symbol",
    "Synonym Symbol",
    "Scientific Name with Author",
    "Common Name",
    "Family",
]


# -------------------------------------------------------------------------
# Try likely delimiters and keep the parse that recovers the expected schema
# -------------------------------------------------------------------------

candidate_separators = [
    "\t",
    "|",
    ",",
]

usda_raw = None
separator_used = None

for sep in candidate_separators:

    try:
        test = pd.read_csv(
            USDA_PLANTS_SOURCE,
            sep=sep,
            dtype="string",
            low_memory=False,
            encoding="utf-8-sig",
        )

        # Normalize column names
        test.columns = (
            test.columns
            .astype(str)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
            .str.strip('"')
            .str.strip("'")
        )

        n_required_found = sum(
            c in test.columns
            for c in required_usda_cols
        )

        print(
            f"Separator {repr(sep)}:"
            f" {len(test.columns)} columns,"
            f" {n_required_found}/{len(required_usda_cols)}"
            " required fields found"
        )

        if n_required_found == len(required_usda_cols):
            usda_raw = test
            separator_used = sep
            break

    except Exception as e:
        print(
            f"Separator {repr(sep)} failed:"
            f" {type(e).__name__}: {e}"
        )


# -------------------------------------------------------------------------
# Fallback: Python delimiter inference
# -------------------------------------------------------------------------

if usda_raw is None:

    print("\nTrying automatic delimiter inference...")

    test = pd.read_csv(
        USDA_PLANTS_SOURCE,
        sep=None,
        engine="python",
        dtype="string",
        encoding="utf-8-sig",
    )

    test.columns = (
        test.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.strip('"')
        .str.strip("'")
    )

    missing = [
        c for c in required_usda_cols
        if c not in test.columns
    ]

    if not missing:
        usda_raw = test
        separator_used = "auto"


# -------------------------------------------------------------------------
# Fail informatively only if no parser recovered the schema
# -------------------------------------------------------------------------

if usda_raw is None:

    raise ValueError(
        "Could not recover the expected USDA PLANTS schema.\n"
        "Inspect the raw header printed above before proceeding."
    )


print("\nUSDA file parsed successfully.")
print("Separator used:", repr(separator_used))
print("Rows:", f"{len(usda_raw):,}")
print("Columns:", len(usda_raw.columns))

print("\nParsed columns:")
for col in usda_raw.columns:
    print(f"  {repr(col)}")


# -------------------------------------------------------------------------
# Final schema check
# -------------------------------------------------------------------------

missing_usda_cols = [
    c for c in required_usda_cols
    if c not in usda_raw.columns
]

assert not missing_usda_cols, (
    "USDA schema still missing: "
    + ", ".join(missing_usda_cols)
)


# -------------------------------------------------------------------------
# Normalize USDA symbol fields
# -------------------------------------------------------------------------

for col in [
    "Symbol",
    "Synonym Symbol",
]:
    usda_raw[col] = (
        usda_raw[col]
        .astype("string")
        .str.strip()
    )

    usda_raw.loc[
        usda_raw[col].eq(""),
        col
    ] = pd.NA


# -------------------------------------------------------------------------
# Separate accepted taxa and synonym records
# -------------------------------------------------------------------------

accepted_rows = usda_raw[
    usda_raw["Synonym Symbol"].isna()
].copy()

synonym_rows = usda_raw[
    usda_raw["Synonym Symbol"].notna()
].copy()


print(
    "\nAccepted-record rows:",
    f"{len(accepted_rows):,}"
)

print(
    "Synonym-record rows:",
    f"{len(synonym_rows):,}"
)

USDA source:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\plantlst.txt

Raw file header:
'"Symbol","Synonym Symbol","Scientific Name with Author","Common Name","Family"\n'
Separator '\t': 1 columns, 0/5 required fields found
Separator '|': 1 columns, 0/5 required fields found
Separator ',': 5 columns, 5/5 required fields found

USDA file parsed successfully.
Separator used: ','
Rows: 93,157
Columns: 5

Parsed columns:
  'Symbol'
  'Synonym Symbol'
  'Scientific Name with Author'
  'Common Name'
  'Family'

Accepted-record rows: 48,994
Synonym-record rows: 44,163


## 6. Build USDA accepted-taxon table

In [96]:
# =========================================================================
# 6. BUILD USDA ACCEPTED-TAXON TABLE
# =========================================================================

usda_accepted = (
    accepted_rows[
        [
            "Symbol",
            "Scientific Name with Author",
            "Common Name",
            "Family",
        ]
    ]
    .dropna(subset=["Symbol"])
    .drop_duplicates()
    .copy()
)


# -------------------------------------------------------------------------
# Check whether any accepted symbol has genuinely conflicting taxonomy
# -------------------------------------------------------------------------

accepted_symbol_counts = (
    usda_accepted
    .groupby("Symbol", dropna=False)
    .size()
)

accepted_conflicts = (
    accepted_symbol_counts[
        accepted_symbol_counts > 1
    ]
)

print(
    "Accepted USDA symbols:",
    f"{usda_accepted['Symbol'].nunique():,}"
)

print(
    "Accepted symbols with >1 distinct metadata row:",
    f"{len(accepted_conflicts):,}"
)

if len(accepted_conflicts) > 0:
    display(
        usda_accepted[
            usda_accepted["Symbol"].isin(
                accepted_conflicts.index
            )
        ]
        .sort_values("Symbol")
        .head(100)
    )


# -------------------------------------------------------------------------
# For lookup purposes, retain one metadata row per accepted symbol.
#
# We are not changing taxonomy here; duplicate identical rows have already
# been removed above. Any remaining conflict is preserved in the diagnostic.
# -------------------------------------------------------------------------

usda_accepted_lookup = (
    usda_accepted
    .drop_duplicates(
        subset="Symbol",
        keep="first"
    )
    .rename(
        columns={
            "Symbol":
                "USDA_accepted_symbol",

            "Scientific Name with Author":
                "USDA_scientific_name",

            "Common Name":
                "USDA_common_name",

            "Family":
                "USDA_family",
        }
    )
    .copy()
)

print(
    "\nAccepted lookup rows:",
    f"{len(usda_accepted_lookup):,}"
)

Accepted USDA symbols: 48,994
Accepted symbols with >1 distinct metadata row: 0

Accepted lookup rows: 48,994


## 7. Build USDA synonym map

In [97]:
# =========================================================================
# 7. BUILD USDA SYNONYM → ACCEPTED SYMBOL MAP
# =========================================================================

usda_synonym_pairs = (
    synonym_rows[
        [
            "Synonym Symbol",
            "Symbol",
        ]
    ]
    .dropna(
        subset=[
            "Synonym Symbol",
            "Symbol",
        ]
    )
    .drop_duplicates()
    .rename(
        columns={
            "Synonym Symbol":
                "USDA_synonym_symbol",

            "Symbol":
                "USDA_accepted_symbol",
        }
    )
    .copy()
)


# -------------------------------------------------------------------------
# Detect synonym symbols pointing to >1 accepted symbol
# -------------------------------------------------------------------------

synonym_target_counts = (
    usda_synonym_pairs
    .groupby("USDA_synonym_symbol")[
        "USDA_accepted_symbol"
    ]
    .nunique()
)

ambiguous_synonyms = (
    synonym_target_counts[
        synonym_target_counts > 1
    ]
)

print(
    "Unique USDA synonym symbols:",
    f"{usda_synonym_pairs['USDA_synonym_symbol'].nunique():,}"
)

print(
    "Ambiguous synonym symbols:",
    f"{len(ambiguous_synonyms):,}"
)


# -------------------------------------------------------------------------
# Exclude only genuinely ambiguous synonym mappings
# -------------------------------------------------------------------------

unambiguous_synonym_pairs = (
    usda_synonym_pairs[
        ~usda_synonym_pairs[
            "USDA_synonym_symbol"
        ].isin(
            ambiguous_synonyms.index
        )
    ]
    .drop_duplicates(
        subset="USDA_synonym_symbol",
        keep="first"
    )
    .copy()
)

print(
    "Usable synonym mappings:",
    f"{len(unambiguous_synonym_pairs):,}"
)

Unique USDA synonym symbols: 44,163
Ambiguous synonym symbols: 0
Usable synonym mappings: 44,163


## 8. Resolve observed codes to USDA taxonomy

In [98]:
# =========================================================================
# 8. RESOLVE OBSERVED CODES TO USDA TAXONOMY
# =========================================================================

# -------------------------------------------------------------------------
# Taxonomy eligibility
#
# Only codes not already identified as protocol/nonplant are eligible.
# Literal NONE is eligible.
# __NO_CANOPY__ is not.
# -------------------------------------------------------------------------

taxonomy_eligible = (
    ~dictionary["source_code_was_blank"]
    & ~dictionary["code_class"].isin(
        [
            "NoCanopy",
            "NonPlant",
            "ContextDependent",
            "ProtocolUnresolved",
        ]
    )
)

print(
    "Taxonomy-eligible codes:",
    f"{taxonomy_eligible.sum():,}"
)


# -------------------------------------------------------------------------
# Build fast lookup structures
# -------------------------------------------------------------------------

accepted_symbols = set(
    usda_accepted_lookup[
        "USDA_accepted_symbol"
    ]
    .dropna()
)

synonym_to_accepted = dict(
    zip(
        unambiguous_synonym_pairs[
            "USDA_synonym_symbol"
        ],
        unambiguous_synonym_pairs[
            "USDA_accepted_symbol"
        ],
    )
)


# -------------------------------------------------------------------------
# Initialize resolution columns
# -------------------------------------------------------------------------

dictionary["USDA_accepted_symbol"] = pd.NA
dictionary["USDA_match_method"] = pd.NA


# -------------------------------------------------------------------------
# 1. Exact accepted-symbol matches
# -------------------------------------------------------------------------

exact_mask = (
    taxonomy_eligible
    & dictionary["observed_code"].isin(
        accepted_symbols
    )
)

dictionary.loc[
    exact_mask,
    "USDA_accepted_symbol"
] = dictionary.loc[
    exact_mask,
    "observed_code"
]

dictionary.loc[
    exact_mask,
    "USDA_match_method"
] = "accepted_symbol"


# -------------------------------------------------------------------------
# 2. Synonym matches among codes still unresolved
# -------------------------------------------------------------------------

synonym_mask = (
    taxonomy_eligible
    & dictionary["USDA_accepted_symbol"].isna()
    & dictionary["observed_code"].isin(
        synonym_to_accepted
    )
)

dictionary.loc[
    synonym_mask,
    "USDA_accepted_symbol"
] = (
    dictionary.loc[
        synonym_mask,
        "observed_code"
    ]
    .map(synonym_to_accepted)
)

dictionary.loc[
    synonym_mask,
    "USDA_match_method"
] = "synonym_symbol"


# -------------------------------------------------------------------------
# Summary before metadata join
# -------------------------------------------------------------------------

print(
    "\nExact accepted-symbol matches:",
    f"{exact_mask.sum():,}"
)

print(
    "Synonym-symbol matches:",
    f"{synonym_mask.sum():,}"
)

print(
    "Total USDA-resolved rows:",
    f"{dictionary['USDA_accepted_symbol'].notna().sum():,}"
)

print(
    "Unique accepted USDA taxa represented:",
    f"{dictionary['USDA_accepted_symbol'].nunique():,}"
)

Taxonomy-eligible codes: 10,937

Exact accepted-symbol matches: 7,160
Synonym-symbol matches: 415
Total USDA-resolved rows: 7,575
Unique accepted USDA taxa represented: 7,264


## 9. Attach USDA taxonomic metadata

In [99]:
# =========================================================================
# 9. ATTACH USDA TAXONOMIC METADATA
# =========================================================================

dictionary = dictionary.merge(
    usda_accepted_lookup,
    on="USDA_accepted_symbol",
    how="left",
    validate="many_to_one",
)


# -------------------------------------------------------------------------
# Any successful USDA taxonomic resolution is a plant
# -------------------------------------------------------------------------

usda_plant_mask = (
    dictionary["USDA_accepted_symbol"].notna()
    & dictionary["code_class"].isna()
)

dictionary.loc[
    usda_plant_mask,
    "code_class"
] = "Plant"


# -------------------------------------------------------------------------
# USDA scientific name becomes label where protocol label is absent
# -------------------------------------------------------------------------

dictionary["resolved_label"] = (
    dictionary["resolved_label"]
    .combine_first(
        dictionary["USDA_scientific_name"]
    )
)


# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

display(
    dictionary["code_class"]
    .value_counts(dropna=False)
    .rename_axis("code_class")
    .reset_index(name="n_codes")
)

,code_class,n_codes
0,Plant,7575
1,<NA>,3362
2,NonPlant,17
3,ContextDependent,3
4,ProtocolUnresolved,2
5,NoCanopy,1


## 10. Sentinel QA: __NO_CANOPY__ versus literal NONE

In [100]:
# =========================================================================
# 10. CRITICAL SENTINEL / USDA NONE QA
# =========================================================================

critical = dictionary[
    dictionary["observed_code"].isin(
        [
            "__NO_CANOPY__",
            "NONE",
        ]
    )
].copy()

display(
    critical[
        [
            "observed_code",
            "observed_code_raw",
            "source_code_was_blank",
            "code_class",
            "resolved_label",
            "USDA_match_method",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_common_name",
            "USDA_family",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
)


no_canopy = dictionary[
    dictionary["observed_code"].eq(
        "__NO_CANOPY__"
    )
]

literal_none = dictionary[
    dictionary["observed_code"].eq(
        "NONE"
    )
]


# -------------------------------------------------------------------------
# These are now legitimate hard invariants
# -------------------------------------------------------------------------

assert len(no_canopy) == 1

assert (
    no_canopy[
        "source_code_was_blank"
    ]
    .all()
)

assert (
    no_canopy[
        "code_class"
    ]
    .eq("NoCanopy")
    .all()
)

assert (
    no_canopy[
        "USDA_accepted_symbol"
    ]
    .isna()
    .all()
)


assert len(literal_none) == 1

assert (
    ~literal_none[
        "source_code_was_blank"
    ]
    .all()
)

assert (
    literal_none[
        "USDA_accepted_symbol"
    ]
    .eq("NONE")
    .all()
)

assert (
    literal_none[
        "code_class"
    ]
    .eq("Plant")
    .all()
)


print(
    "\nCritical QA passed:"
    "\n  __NO_CANOPY__ is excluded from USDA taxonomy"
    "\n  literal NONE is retained as a USDA plant"
)

,observed_code,observed_code_raw,source_code_was_blank,code_class,resolved_label,USDA_match_method,USDA_accepted_symbol,USDA_scientific_name,USDA_common_name,USDA_family,n_records,n_top,n_lower,n_soil_surface
15,__NO_CANOPY__,<NA>,True,NoCanopy,No top-canopy contact,<NA>,<NA>,<NA>,<NA>,<NA>,442612,442612,0,0
2527,NONE,NONE,False,Plant,Notholaena neglecta Maxon,accepted_symbol,NONE,Notholaena neglecta Maxon,Maxon's cloak fern,Pteridaceae,398,394,4,0



Critical QA passed:
  __NO_CANOPY__ is excluded from USDA taxonomy
  literal NONE is retained as a USDA plant


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_19040\2614387556.py:82: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  ~literal_none[


## 11. Resolve generic numbered vegetation codes

In [101]:
# =========================================================================
# 11. RESOLVE GENERIC NUMBERED VEGETATION CODES
# =========================================================================

generic_rules = {
    "AF": ("Annual forb", "AnnualForb"),
    "PF": ("Perennial forb", "PerennialForb"),
    "AG": ("Annual graminoid", "AnnualGraminoid"),
    "PG": ("Perennial graminoid", "PerennialGraminoid"),
    "SH": ("Shrub", "Shrub"),
    "TR": ("Tree", "Tree"),
}

dictionary["protocol_plant_group"] = pd.Series(
    pd.NA,
    index=dictionary.index,
    dtype="string"
)

dictionary["generic_code_prefix"] = pd.Series(
    pd.NA,
    index=dictionary.index,
    dtype="string"
)

for prefix, (label, group) in generic_rules.items():

    # Must have at least one digit after the prefix.
    # Therefore bare "AG" is NOT captured.
    pattern = rf"^{prefix}\d+$"

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].str.match(
            pattern,
            na=False
        )
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group
    dictionary.loc[mask, "generic_code_prefix"] = prefix


generic_resolved = dictionary[
    dictionary["generic_code_prefix"].notna()
].copy()

print(
    "Generic numbered plant codes resolved:",
    f"{len(generic_resolved):,}"
)

print(
    "Records represented:",
    f"{generic_resolved['n_records'].sum():,}"
)

display(
    generic_resolved[
        [
            "observed_code",
            "resolved_label",
            "protocol_plant_group",
            "n_records",
            "n_top",
            "n_lower",
            "n_soil_surface",
        ]
    ]
    .sort_values("n_records", ascending=False)
    .head(100)
)

Generic numbered plant codes resolved: 2,954
Records represented: 40,570


,observed_code,resolved_label,protocol_plant_group,n_records,n_top,n_lower,n_soil_surface
128,SH00,Shrub,Shrub,4187,3145,691,351
156,AF00,Annual forb,AnnualForb,2058,1266,786,6
214,PF00,Perennial forb,PerennialForb,1268,825,435,8
724,TR00,Tree,Tree,972,878,67,27
459,PG00,Perennial graminoid,PerennialGraminoid,687,408,173,106
762,AG04001,Annual graminoid,AnnualGraminoid,567,546,21,0
532,AF01000,Annual forb,AnnualForb,528,386,141,1
579,AF02000,Annual forb,AnnualForb,498,347,149,2
791,PG01,Perennial graminoid,PerennialGraminoid,456,269,98,89
661,SH03000,Shrub,Shrub,319,254,40,25


## 12. Resolve explicit descriptive vegetation classes

In [102]:
# =========================================================================
# 12. RESOLVE EXPLICIT DESCRIPTIVE VEGETATION CLASSES
# =========================================================================

descriptive_plant_rules = {
    "Perennial grasses": (
        "Perennial grasses",
        "PerennialGraminoid"
    ),

    "Annual plants": (
        "Annual plants",
        "AnnualPlant"
    ),

    "Shrubs": (
        "Shrubs",
        "Shrub"
    ),

    "Sub-shrubs and perennial forbs": (
        "Sub-shrubs and perennial forbs",
        "SubshrubOrPerennialForb"
    ),

    "Trees": (
        "Trees",
        "Tree"
    ),
}

for code, (label, group) in descriptive_plant_rules.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(code)
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group

## 13. Consolidated protocol/project/manual code resolution

This is the existing reviewed rule registry. It remains intentionally explicit and verbose because these are domain decisions, not generic parsing rules. The key constraint is unchanged: it may classify unresolved codes, but it must never overwrite `observed_code`.


In [103]:
# ============================================================
# CONSOLIDATED LPI CODE RESOLUTION
#
# Run AFTER:
#   1. raw code canonicalization
#   2. core protocol classification
#   3. USDA accepted-symbol + synonym resolution
#
# This cell:
#   - preserves observed_code exactly
#   - only modifies currently unresolved codes
#   - handles protocol/material codes
#   - handles functional/project codes
#   - handles project-code regex families
#   - handles manual taxonomic aliases
#   - preserves uncertainty/provenance
# ============================================================

import re
import pandas as pd


# ============================================================
# 0. ENSURE REQUIRED OUTPUT COLUMNS EXIST
# ============================================================

required_resolution_columns = [
    "resolved_label",
    "resolution_source",
    "taxonomic_level",
    "protocol_plant_group",
    "USDA_match_method",
    "USDA_accepted_symbol",
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
]

for col in required_resolution_columns:
    if col not in dictionary.columns:
        dictionary[col] = pd.NA


# ============================================================
# 1. FIXED PROTOCOL / MATERIAL / FUNCTIONAL RESOLUTIONS
#
# Format:
# observed_code:
#     resolved_label,
#     code_class,
#     protocol_plant_group,
#     resolution_source
# ============================================================

fixed_rules = {

    # --------------------------------------------------------
    # No-canopy sentinel
    # --------------------------------------------------------
    "__NO_CANOPY__": (
        "No canopy contact",
        "NoCanopy",
        pd.NA,
        "source_blank_sentinel",
    ),

    # --------------------------------------------------------
    # Standard / well-supported non-plant protocol codes
    # --------------------------------------------------------
    "S": (
        "Soil",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "R": (
        "Rock",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "BR": (
        "Bedrock",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "GR": (
        "Gravel",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "CB": (
        "Cobble",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "ST": (
        "Stone",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "BY": (
        "Boulder",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "RF": (
        "Rock fragment",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "FG": (
        "Fine gravel",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "HL": (
        "Herbaceous litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "WL": (
        "Woody litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "EL": (
        "Embedded litter",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "D": (
        "Duff",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "Duff": (
        "Duff",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    # IMPORTANT:
    # L = lichen, NOT litter
    "L": (
        "Lichen",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "LC": (
        "Lichen / biological crust",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "M": (
        "Moss",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "CY": (
        "Cyanobacterial crust",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "LM": (
        "Loose erodible mineral soil",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "DN": (
        "Dung",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "AM": (
        "Unattached animal material",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    "HT": (
        "Human trash",
        "NonPlant",
        pd.NA,
        "protocol_code",
    ),

    # --------------------------------------------------------
    # Additional project/material aliases
    # --------------------------------------------------------
    "2MOSS": (
        "Moss",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "MOSS": (
        "Moss",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LW": (
        "Woody litter",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LICHN": (
        "Lichen",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "LIVR86": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "LVRWORT": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2LVRWRT": (
        "Liverwort",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2FUNGI": (
        "Fungi",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "2ALGA": (
        "Algae",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    "ALGAE86": (
        "Algae",
        "NonPlant",
        pd.NA,
        "manual_protocol_alias",
    ),

    # Deposited soil interpretation
    "DS": (
        "Deposited soil",
        "NonPlant",
        pd.NA,
        "manual_protocol_interpretation",
    ),

    # Unknown material codes whose observed vertical placement
    # strongly suggests non-vegetation rather than plant taxa.
    "ER": (
        "Unknown lower-layer material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "CM": (
        "Unknown surface material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "P": (
        "Unknown surface material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    "NT": (
        "Unknown lower-layer material",
        "NonPlant",
        pd.NA,
        "manual_context_resolution",
    ),

    # --------------------------------------------------------
    # Basal vegetation
    # --------------------------------------------------------
    # SoilSurface DOES NOT imply non-plant.
    "Plant base": (
        "Plant base",
        "Plant",
        "UnknownPlant",
        "protocol_basal_vegetation",
    ),

    # --------------------------------------------------------
    # Explicit functional/project vegetation codes
    # --------------------------------------------------------
    "PPGG": (
        "Perennial graminoid",
        "Plant",
        "PerennialGraminoid",
        "manual_functional_resolution",
    ),

    "PPFF": (
        "Perennial forb",
        "Plant",
        "PerennialForb",
        "manual_functional_resolution",
    ),

    "AAFF": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "AAGG": (
        "Annual graminoid",
        "Plant",
        "AnnualGraminoid",
        "manual_functional_resolution",
    ),

    "SHRUB": (
        "Shrub",
        "Plant",
        "Shrub",
        "manual_functional_resolution",
    ),

    "Sh01999": (
        "Shrub",
        "Plant",
        "Shrub",
        "manual_functional_resolution",
    ),

    # SU00 interpreted as Shrub Unknown
    "SU00": (
        "Unknown shrub",
        "Plant",
        "Shrub",
        "manual_project_code_inferred",
    ),

    "UKGR": (
        "Unknown grass",
        "Plant",
        "Graminoid",
        "manual_functional_resolution",
    ),

    "ASTERA": (
        "Aster / unknown forb",
        "Plant",
        "Forb",
        "manual_functional_resolution",
    ),

    "Af02": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "LOTUSAF": (
        "Annual forb",
        "Plant",
        "AnnualForb",
        "manual_functional_resolution",
    ),

    "BRASS2PF": (
        "Perennial Brassicaceae",
        "Plant",
        "PerennialForb",
        "manual_functional_resolution",
    ),

    "POA*": (
        "Graminoid",
        "Plant",
        "Graminoid",
        "manual_functional_resolution",
    ),

    # Explicitly unknown taxonomically but clearly vegetation
    "BOETRII": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "PINSCO": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "Unk": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),

    "UNK": (
        "Unknown plant",
        "Plant",
        "UnknownPlant",
        "manual_unknown_plant",
    ),
}


# ============================================================
# 2. APPLY FIXED RULES
# ============================================================

for code, (
    label,
    code_class,
    plant_group,
    source,
) in fixed_rules.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(code)
    )

    if not mask.any():
        continue

    dictionary.loc[mask, "code_class"] = code_class
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = plant_group
    dictionary.loc[mask, "resolution_source"] = source


# ============================================================
# 3. GENERIC NUMBERED AIM / PROJECT FUNCTIONAL CODES
#
# Strict patterns only.
# Bare AG is intentionally NOT handled here because AG can
# represent a SoilSurface aggregate code in some protocols.
# ============================================================

project_functional_patterns = [
    (r"^AF\d+$",      "Annual forb",         "AnnualForb"),
    (r"^PF\d+$",      "Perennial forb",      "PerennialForb"),
    (r"^AG\d+$",      "Annual graminoid",    "AnnualGraminoid"),
    (r"^PG\d+$",      "Perennial graminoid", "PerennialGraminoid"),
    (r"^SH\d+$",      "Shrub",               "Shrub"),
    (r"^TR\d+$",      "Tree",                "Tree"),

    # Project-prefixed variants
    (r"^2FA\d+$",     "Annual forb",         "AnnualForb"),
    (r"^2FP\d+$",     "Perennial forb",      "PerennialForb"),
    (r"^2GA\d+$",     "Annual graminoid",    "AnnualGraminoid"),
    (r"^2GP\d+$",     "Perennial graminoid", "PerennialGraminoid"),
    (r"^2SHRUB\d+$",  "Shrub",               "Shrub"),
    (r"^2TREE\d+$",   "Tree",                "Tree"),
    (r"^2SUBS\d+$",   "Subshrub",            "Subshrub"),
]


for pattern, label, group in project_functional_patterns:

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].str.match(pattern, na=False)
    )

    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = label
    dictionary.loc[mask, "protocol_plant_group"] = group
    dictionary.loc[mask, "resolution_source"] = "project_code_pattern"


# ============================================================
# 4. SPACED PG PROJECT CODES
#
# PG 04, PG 05, PG 06, etc.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].str.match(
        r"^PG\s+\d+$",
        na=False,
    )
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Perennial graminoid"
dictionary.loc[mask, "protocol_plant_group"] = "PerennialGraminoid"
dictionary.loc[mask, "resolution_source"] = "project_code_pattern"


# ============================================================
# 5. ALL UN... CODES = UNKNOWN PLANT
#
# Preserve the original observed code.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].str.match(
        r"^UN",
        case=False,
        na=False,
    )
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Unknown plant"
dictionary.loc[mask, "protocol_plant_group"] = "UnknownPlant"
dictionary.loc[mask, "resolution_source"] = "project_unknown_code"


# ============================================================
# 6. ELELLA
#
# Observer ambiguity between Elymus elymoides and E. lanceolatus.
# Resolve only to Elymus / perennial graminoid.
# ============================================================

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].eq("ELELLA")
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Elymus"
dictionary.loc[mask, "taxonomic_level"] = "genus"
dictionary.loc[mask, "protocol_plant_group"] = "PerennialGraminoid"
dictionary.loc[mask, "resolution_source"] = (
    "manual_ambiguous_species_to_genus"
)


# ============================================================
# 7. MANUAL TAXONOMIC ALIASES
#
# target = canonical USDA symbol we want to query/join.
#
# The original observed_code is NEVER overwritten.
# ============================================================

manual_alias_candidates = {

    # --------------------------------------------------------
    # Cryptantha variants
    # --------------------------------------------------------
    "CRYPTA": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CRYPT1": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CRYPTa": {
        "target": "CRYPT",
        "label": "Cryptantha",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # --------------------------------------------------------
    # Genus aliases
    # --------------------------------------------------------
    "ALLIUM": {
        "target": "ALLIU",
        "label": "Allium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "ASTRAAF": {
        "target": "ASTRA",
        "label": "Astragalus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEA": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEAF": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "PHACEa": {
        "target": "PHACE",
        "label": "Phacelia",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CYPERA": {
        "target": "CYPER",
        "label": "Cyperus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_typo_genus",
    },

    "ALCOC": {
        "target": "CALOC",
        "label": "Calochortus",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "ERITRA": {
        "target": "ERITR",
        "label": "Eritrichium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "CERASTIUM": {
        "target": "CERAS",
        "label": "Cerastium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # --------------------------------------------------------
    # Gutierrezia cf. sarothrae
    # Preserve cf. uncertainty in resolved label.
    # --------------------------------------------------------
    "GUTCFSAR": {
        "target": "GUSA2",
        "label": "Gutierrezia cf. sarothrae",
        "taxonomic_level": "species_cf",
        "resolution_source": "manual_cf_taxonomic_alias",
    },

    "GUTcfSAR": {
        "target": "GUSA2",
        "label": "Gutierrezia cf. sarothrae",
        "taxonomic_level": "species_cf",
        "resolution_source": "manual_cf_taxonomic_alias",
    },

    # --------------------------------------------------------
    # CHJU project variants
    # --------------------------------------------------------
    "CHJUMG": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    "CHJUR": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    "CHJUMT": {
        "target": "CHJU",
        "label": "CHJU",
        "taxonomic_level": "project_base_code",
        "resolution_source": "manual_project_code_alias",
    },

    # --------------------------------------------------------
    # Species / typo / capitalization corrections
    # --------------------------------------------------------
    "CHIV8": {
        "target": "CHVI8",
        "label": "Chrysothamnus viscidiflorus",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    "Elel5": {
        "target": "ELEL5",
        "label": "Elymus elymoides",
        "taxonomic_level": "species",
        "resolution_source": "manual_case_correction",
    },

    "DRAR86": {
        "target": "POARA4",
        "label": "Potentilla arguta",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "SYOR22": {
        "target": "SYOR",
        "label": "Symphoricarpos orbiculatus",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "ANDI": {
        "target": "ANDI2",
        "label": "Antennaria dimorpha",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    "AMAL22": {
        "target": "AMAL2",
        "label": "Amelanchier alnifolia",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    "ERNA1": {
        "target": "ERNA10",
        "label": "ERNA10",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    "Koma": {
        "target": "KOMA",
        "label": "Koeleria macrantha",
        "taxonomic_level": "species",
        "resolution_source": "manual_case_correction",
    },

    "BAPRV": {
        "target": "BAPR",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "ARTW8": {
        "target": "ARTRW8",
        "label": "Artemisia tridentata ssp. wyomingensis",
        "taxonomic_level": "subspecies",
        "resolution_source": "manual_typo_correction",
    },

    "POAR2R2": {
        "target": "PONIN",
        "label": "Potentilla nivea var. nivea",
        "taxonomic_level": "variety",
        "resolution_source": "manual_USDA_synonym_resolution",
    },

    "SPAL86": {
        "target": "SPAL",
        "label": "Spartina",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    "SPAL_01": {
        "target": "SPAL",
        "label": "Spartina",
        "taxonomic_level": "genus",
        "resolution_source": "manual_project_code_alias",

    # Sphaeralcea coccinea
    "SPCO86": {
        "target": "SPCO",
        "label": "Sphaeralcea coccinea",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    # Bassia prostrata variants
    "BAPRG": {
        "target": "BAPR5",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    "BAPRV": {
        "target": "BAPR5",
        "label": "Bassia prostrata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_alias",
    },

    # Ribes genus
    "RIBE5": {
        "target": "RIBES",
        "label": "Ribes",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # Linum lewisii
    # observed LILEL3 -> likely intended LILE3
    "LILEL3": {
        "target": "LILE3",
        "label": "Linum lewisii",
        "taxonomic_level": "species",
        "resolution_source": "manual_typo_correction",
    },

    # Lepidium genus
    "LEPEAS": {
        "target": "LEPE",
        "label": "Lepidium",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },

    # Comandra umbellata
    "COUM3": {
        "target": "COUM3",
        "label": "Comandra umbellata",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    # Phlox longifolia
    "PHLO22": {
        "target": "PHLO22",
        "label": "Phlox longifolia",
        "taxonomic_level": "species",
        "resolution_source": "manual_taxonomic_resolution",
    },

    # Penstemon genus
    "PENST4": {
        "target": "PENST",
        "label": "Penstemon",
        "taxonomic_level": "genus",
        "resolution_source": "manual_genus_alias",
    },
    },
}


# ============================================================
# 8. HELPER: ROBUST USDA TARGET LOOKUP
#
# Supports the column naming convention used in the current
# usda_accepted_lookup table.
# ============================================================

def get_usda_target_row(target):
    """
    Return exactly one USDA accepted-symbol row if available.
    Otherwise return None.
    """

    if "usda_accepted_lookup" not in globals():
        return None

    if "USDA_accepted_symbol" not in usda_accepted_lookup.columns:
        return None

    hit = usda_accepted_lookup[
        usda_accepted_lookup["USDA_accepted_symbol"].eq(target)
    ]

    if len(hit) == 1:
        return hit.iloc[0]

    return None


# ============================================================
# 9. APPLY MANUAL TAXONOMIC ALIASES
# ============================================================

alias_warnings = []

for observed_code, info in manual_alias_candidates.items():

    mask = (
        dictionary["code_class"].isna()
        & dictionary["observed_code"].eq(observed_code)
    )

    if not mask.any():
        continue

    target = info["target"]

    # We know these represent plants even if USDA target lookup fails.
    dictionary.loc[mask, "code_class"] = "Plant"
    dictionary.loc[mask, "resolved_label"] = info["label"]
    dictionary.loc[mask, "resolution_source"] = info["resolution_source"]
    dictionary.loc[mask, "taxonomic_level"] = info["taxonomic_level"]
    dictionary.loc[mask, "USDA_accepted_symbol"] = target
    dictionary.loc[mask, "USDA_match_method"] = "manual_alias"

    if "protocol_plant_group" in info:
        dictionary.loc[
            mask,
            "protocol_plant_group"
        ] = info["protocol_plant_group"]

    # Attach USDA metadata when target is found locally
    target_row = get_usda_target_row(target)

    if target_row is not None:

        metadata_cols = [
            "USDA_scientific_name",
            "USDA_common_name",
            "USDA_family",
        ]

        for col in metadata_cols:
            if col in target_row.index:
                dictionary.loc[mask, col] = target_row[col]

    else:
        alias_warnings.append(
            (observed_code, target)
        )


# ============================================================
# 10. OPTIONAL MANUAL FUNCTIONAL OVERRIDES FOR ALIASED TAXA
#
# These encode ecological information that is defensible even
# when the taxonomic alias itself is genus-level or uncertain.
# ============================================================

functional_overrides = {

    # Elymus ambiguity resolved structurally
    "ELELLA": "PerennialGraminoid",

    # Unknown grasses / graminoids
    "UKGR": "Graminoid",

    # Aster only known as forb
    "ASTERA": "Forb",

    # Explicit project functional encodings
    "LOTUSAF": "AnnualForb",
    "BRASS2PF": "PerennialForb",

    # Wyoming big sagebrush
    "ARTW8": "Shrub",

    # Koeleria
    "Koma": "PerennialGraminoid",

    # Cordgrass / Spartina
    "SPAL86": "PerennialGraminoid",
    "SPAL_01": "PerennialGraminoid",

    # Bassia prostrata:
    # treat as exotic forb in this framework, NOT shrub
    "BAPRV": "Forb",
    "BAPRG": "Forb",

    # Sphaeralcea coccinea
    "SPCO86": "Forb",

    # Ribes
    "RIBE5": "Shrub",

    # Linum lewisii
    "LILEL3": "Forb",

    # Lepidium
    "LEPEAS": "Forb",

    # Comandra umbellata
    "COUM3": "Forb",

    # Phlox longifolia
    "PHLO22": "Forb",

    # Penstemon
    "PENST4": "Forb",
}


for code, group in functional_overrides.items():

    mask = dictionary["observed_code"].eq(code)

    dictionary.loc[
        mask,
        "protocol_plant_group"
    ] = group


# ============================================================
# 11. QA: MANUAL RULES THAT FAILED USDA TARGET LOOKUP
#
# This does NOT undo their classification.
# It simply tells us which canonical targets were not found
# in the local accepted-symbol lookup.
# ============================================================

if alias_warnings:

    alias_warning_df = pd.DataFrame(
        alias_warnings,
        columns=[
            "observed_code",
            "requested_USDA_target",
        ],
    ).drop_duplicates()

    print(
        f"Manual alias targets not found in "
        f"usda_accepted_lookup: {len(alias_warning_df):,}"
    )

    display(alias_warning_df)

else:
    print(
        "All applied manual alias targets found "
        "in usda_accepted_lookup."
    )


# ============================================================
# 12. QA: DISPLAY ALL CODES TOUCHED BY THIS CONSOLIDATED CELL
# ============================================================

manual_codes = (
    set(fixed_rules.keys())
    | set(manual_alias_candidates.keys())
)

qa_manual = (
    dictionary[
        dictionary["observed_code"].isin(manual_codes)
        | dictionary["resolution_source"].isin(
            [
                "project_code_pattern",
                "project_unknown_code",
                "manual_ambiguous_species_to_genus",
            ]
        )
    ][
        [
            "observed_code",
            "n_records",
            "code_class",
            "resolved_label",
            "USDA_accepted_symbol",
            "protocol_plant_group",
            "taxonomic_level",
            "resolution_source",
        ]
    ]
    .sort_values(
        ["n_records", "observed_code"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    f"Codes represented in consolidated resolution QA: "
    f"{len(qa_manual):,}"
)

display(qa_manual)


# ============================================================
# 13. FINAL UNRESOLVED TABLE
# ============================================================

unresolved = (
    dictionary[
        dictionary["code_class"].isna()
    ][
        [
            "observed_code",
            "n_records",
            "n_plot_visits",
            "n_top",
            "n_lower",
            "n_soil_surface",
            "first_year",
            "last_year",
        ]
    ]
    .sort_values(
        "n_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f"Remaining unresolved codes: "
    f"{len(unresolved):,}"
)

print(
    f"Records represented: "
    f"{unresolved['n_records'].sum():,}"
)

display(unresolved.head(100))


# ============================================================
# 14. STOPPING-RULE QA
#
# Our manual-review threshold is 50 observations.
# Anything remaining below this threshold can be retained as
# unresolved rather than forcing increasingly speculative IDs.
# ============================================================

needs_review = unresolved[
    unresolved["n_records"] >= 50
].copy()

below_manual_threshold = unresolved[
    unresolved["n_records"] < 50
].copy()

print()
print(
    f"Still >= 50 records and requiring review: "
    f"{len(needs_review):,}"
)

print(
    f"Remaining < 50-record codes: "
    f"{len(below_manual_threshold):,}"
)

print(
    f"Records represented by <50 tail: "
    f"{below_manual_threshold['n_records'].sum():,}"
)

display(needs_review)

All applied manual alias targets found in usda_accepted_lookup.
Codes represented in consolidated resolution QA: 246


,observed_code,n_records,code_class,resolved_label,USDA_accepted_symbol,protocol_plant_group,taxonomic_level,resolution_source
0,S,4908062,NonPlant,Soil,<NA>,<NA>,<NA>,<NA>
1,HL,2420983,NonPlant,Herbaceous litter,<NA>,<NA>,<NA>,<NA>
2,L,853273,NonPlant,Lichen,<NA>,<NA>,<NA>,<NA>
3,GR,721055,NonPlant,Gravel,<NA>,<NA>,<NA>,<NA>
4,__NO_CANOPY__,442612,NoCanopy,No top-canopy contact,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
241,UN23070,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
242,UN25037,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
243,UN33009,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code
244,UNK10,1,Plant,Unknown plant,<NA>,UnknownPlant,<NA>,project_unknown_code


Remaining unresolved codes: 172
Records represented: 2,186


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
0,PHYGOR,365,50,271,89,5,2018,2023
1,PHYFEN,290,66,193,90,7,2018,2023
2,SIDTEN,246,25,227,17,2,2018,2024
3,SPOCFCRY,220,38,173,29,18,2018,2021
4,BOERHAF,75,10,72,3,0,2022,2023
5,PS09009,63,1,45,12,6,2023,2023
6,PS91006,57,2,41,15,1,2021,2021
7,SPHPUM,39,24,24,13,2,2018,2023
8,PL,33,10,1,21,11,2018,2018
9,DF,25,13,2,23,0,2021,2022



Still >= 50 records and requiring review: 7
Remaining < 50-record codes: 165
Records represented by <50 tail: 870


,observed_code,n_records,n_plot_visits,n_top,n_lower,n_soil_surface,first_year,last_year
0,PHYGOR,365,50,271,89,5,2018,2023
1,PHYFEN,290,66,193,90,7,2018,2023
2,SIDTEN,246,25,227,17,2,2018,2024
3,SPOCFCRY,220,38,173,29,18,2018,2021
4,BOERHAF,75,10,72,3,0,2022,2023
5,PS09009,63,1,45,12,6,2023,2023
6,PS91006,57,2,41,15,1,2021,2021


## 14. Finalize manually reviewed unknown-plant codes

In [104]:
# ============================================================
# FINALIZE REMAINING HIGH-FREQUENCY UNRESOLVED PLANT CODES
#
# These have been manually reviewed and cannot be resolved
# defensibly beyond unknown vegetation.
# ============================================================

reviewed_unknown_plant_codes = [
    "PHYGOR",
    "PHYFEN",
    "SIDTEN",
    "SPOCFCRY",
    "BOERHAF",
    "PS09009",
    "PS91006",
]

mask = (
    dictionary["code_class"].isna()
    & dictionary["observed_code"].isin(reviewed_unknown_plant_codes)
)

dictionary.loc[mask, "code_class"] = "Plant"
dictionary.loc[mask, "resolved_label"] = "Unknown plant"
dictionary.loc[mask, "protocol_plant_group"] = "UnknownPlant"
dictionary.loc[mask, "resolution_source"] = "manual_review_unresolvable"

## 15. Backfill resolution provenance

In [105]:
# ============================================================
# BACKFILL RESOLUTION PROVENANCE
#
# Earlier pipeline stages resolved many codes before
# resolution_source was introduced. This reconstructs
# provenance without overwriting later/manual decisions.
# ============================================================

source_missing = (
    dictionary["resolution_source"].isna()
    | dictionary["resolution_source"].astype("string").str.strip().eq("")
)


# ------------------------------------------------------------
# 1. No-canopy sentinel
# ------------------------------------------------------------

mask = (
    source_missing
    & dictionary["observed_code"].eq("__NO_CANOPY__")
)

dictionary.loc[
    mask,
    "resolution_source"
] = "source_blank_sentinel"


# ------------------------------------------------------------
# 2. USDA taxonomy resolution
#
# Prefer USDA_match_method because it preserves whether the
# original resolution was accepted-symbol vs synonym-symbol.
# ------------------------------------------------------------

if "USDA_match_method" in dictionary.columns:

    usda_exact = (
        source_missing
        & dictionary["USDA_match_method"].astype("string").isin([
            "accepted",
            "accepted_symbol",
            "exact",
            "exact_accepted_symbol",
        ])
    )

    dictionary.loc[
        usda_exact,
        "resolution_source"
    ] = "USDA_accepted_symbol"


    usda_synonym = (
        source_missing
        & dictionary["USDA_match_method"].astype("string").isin([
            "synonym",
            "synonym_symbol",
            "USDA_synonym",
        ])
    )

    dictionary.loc[
        usda_synonym,
        "resolution_source"
    ] = "USDA_synonym_symbol"


# Catch USDA-resolved rows whose older match-method naming
# does not match the strings above.
source_missing = dictionary["resolution_source"].isna()

mask = (
    source_missing
    & dictionary["USDA_accepted_symbol"].notna()
)

dictionary.loc[
    mask,
    "resolution_source"
] = "USDA_taxonomic_resolution"


# ------------------------------------------------------------
# 3. Previously resolved protocol/nonplant codes
#
# Only fills remaining missing provenance.
# ------------------------------------------------------------

source_missing = dictionary["resolution_source"].isna()

core_protocol_codes = {
    "S", "R", "BR", "GR", "CB", "ST", "BY",
    "RF", "FG",
    "HL", "WL", "EL", "D",
    "L", "LC", "M", "CY",
    "LM", "DN", "AM", "HT",
}

mask = (
    source_missing
    & dictionary["observed_code"].isin(core_protocol_codes)
)

dictionary.loc[
    mask,
    "resolution_source"
] = "protocol_code"


# ------------------------------------------------------------
# 4. Existing resolved rows lacking finer provenance
#
# This is deliberately generic. It prevents resolved rows from
# appearing as "unresolved" while retaining the fact that the
# exact historical resolution pathway was not recorded.
# ------------------------------------------------------------

source_missing = dictionary["resolution_source"].isna()

mask = (
    source_missing
    & dictionary["code_class"].notna()
)

dictionary.loc[
    mask,
    "resolution_source"
] = "legacy_resolved_pre_provenance"


# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print(
    dictionary["resolution_source"]
    .value_counts(dropna=False)
)

still_missing = dictionary[
    dictionary["resolution_source"].isna()
]

print(
    f"\nRows still lacking resolution_source: "
    f"{len(still_missing):,}"
)

resolution_source
USDA_accepted_symbol                 7160
legacy_resolved_pre_provenance       2967
USDA_synonym_symbol                   415
<NA>                                  165
project_unknown_code                  130
project_code_pattern                   29
protocol_code                          22
manual_functional_resolution           12
manual_genus_alias                     12
manual_protocol_alias                  10
manual_review_unresolvable              7
manual_project_code_alias               4
manual_unknown_plant                    4
manual_context_resolution               4
manual_typo_correction                  3
manual_taxonomic_alias                  3
manual_cf_taxonomic_alias               2
manual_taxonomic_resolution             2
manual_case_correction                  2
source_blank_sentinel                   1
protocol_basal_vegetation               1
manual_project_code_inferred            1
manual_protocol_interpretation          1
manual_ambiguous

## 16. Known taxonomic correction before ecological-trait merge

`POAR2R2` is retained as the observed field code but resolves to USDA accepted symbol `PONIN`. This correction occurs **before** ecological traits are joined so the trait merge follows the accepted symbol.


In [106]:
# ============================================================================
# KNOWN TAXONOMIC CORRECTION: POAR2R2 -> PONIN
# ============================================================================

mask = dictionary["observed_code"].eq("POAR2R2")

assert mask.sum() == 1, (
    f"Expected one POAR2R2 dictionary row; found {mask.sum()}."
)

idx = dictionary.index[mask][0]

# Preserve the raw observed code and correct only the accepted USDA identity.
dictionary.loc[idx, "USDA_accepted_symbol"] = "PONIN"
dictionary.loc[idx, "USDA_match_method"] = "manual_synonym_resolution"
dictionary.loc[idx, "resolution_source"] = "manual_USDA_synonym_resolution"
dictionary.loc[idx, "code_class"] = "Plant"

# Pull accepted taxonomy from the authoritative accepted-taxon lookup that
# was actually created in Section 6.
ponin = usda_accepted_lookup.loc[
    usda_accepted_lookup["USDA_accepted_symbol"].eq("PONIN")
].copy()

assert len(ponin) == 1, (
    f"Expected exactly one accepted USDA row for PONIN; found {len(ponin)}."
)

ponin = ponin.iloc[0]

for col in [
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
]:
    if col in dictionary.columns and col in ponin.index:
        dictionary.loc[idx, col] = ponin[col]

# Prefer the accepted USDA scientific name as the resolved label.
if pd.notna(ponin.get("USDA_scientific_name")):
    dictionary.loc[idx, "resolved_label"] = ponin["USDA_scientific_name"]
else:
    dictionary.loc[idx, "resolved_label"] = "Potentilla nivea var. nivea"

# Hard checks:
# - observed provenance stays intact
# - accepted identity is corrected
assert dictionary.loc[idx, "observed_code"] == "POAR2R2"
assert dictionary.loc[idx, "USDA_accepted_symbol"] == "PONIN"

print(
    "POAR2R2 preserved as observed_code; "
    "accepted USDA identity corrected to PONIN."
)

display(
    dictionary.loc[
        [idx],
        [
            "observed_code",
            "USDA_accepted_symbol",
            "USDA_match_method",
            "USDA_scientific_name",
            "USDA_common_name",
            "USDA_family",
            "resolved_label",
            "resolution_source",
        ],
    ]
)

POAR2R2 preserved as observed_code; accepted USDA identity corrected to PONIN.


,observed_code,USDA_accepted_symbol,USDA_match_method,USDA_scientific_name,USDA_common_name,USDA_family,resolved_label,resolution_source
10143,POAR2R2,PONIN,manual_synonym_resolution,Potentilla nivea L. var. nivea,snow cinquefoil,Rosaceae,Potentilla nivea L. var. nivea,manual_USDA_synonym_resolution


## 17. Audit and complete the USDA ecological-trait cache

The ecological cache is not assumed to be perfect. This section derives the exact USDA accepted-symbol universe required by the current dictionary, then audits the cache against it.

A profile is queried when either:

1. its accepted symbol is absent from the cache; or
2. the cached row is incomplete and has **not** already been verified with the validated root-level parser.

The parser reads only the root `PlantProfile` fields (`Durations`, `GrowthHabits`, `NativeStatuses`). It never recursively traverses `Ancestors`, which was the source of the earlier null-trait corruption.

Successfully queried profiles are marked with `USDA_profile_verified_root_parser = True`. Therefore, a taxon that legitimately lacks a USDA trait is not queried on every future run.

If an API call fails, the taxon is retained in the dictionary and will fall back to whatever structural/unknown interpretation is supported by the available information. Failed symbols are written to a recovery-failure CSV and retried on a later run.


In [107]:
# ============================================================================
# 17. AUDIT + COMPLETE USDA ECOLOGICAL-TRAIT CACHE
# ============================================================================

# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def clean_text(value):
    if value is None or pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "" or value.lower() in {
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return pd.NA

    return value


def unique_pipe(values):
    """Preserve multiple USDA values in stable pipe-delimited form."""
    if values is None:
        return pd.NA

    if not isinstance(values, (list, tuple, set)):
        values = [values]

    out = []

    for value in values:
        value = clean_text(value)

        if pd.notna(value) and value not in out:
            out.append(value)

    return "|".join(out) if out else pd.NA


def canonical_l48_status(value):
    """
    Canonical status field:
        N = Native
        I = Introduced

    Historical cache rows may contain N/I or Native/Introduced.
    """
    if value is None or pd.isna(value):
        return pd.NA

    parts = {
        x.strip().lower()
        for x in str(value).split("|")
        if x.strip()
    }

    out = []

    # Keep introduced first if both are ever returned.
    if parts.intersection({
        "i",
        "introduced",
        "non-native",
        "nonnative",
        "exotic",
    }):
        out.append("I")

    if parts.intersection({
        "n",
        "native",
    }):
        out.append("N")

    return "|".join(out) if out else pd.NA


def canonical_l48_type(status):
    if pd.isna(status):
        return pd.NA

    tokens = set(str(status).split("|"))

    out = []

    if "I" in tokens:
        out.append("Introduced")

    if "N" in tokens:
        out.append("Native")

    return "|".join(out) if out else pd.NA


def parse_profile_payload(symbol, payload):
    """
    Parse ONLY the root USDA PlantProfile object.

    Root fields used:
        Id
        Symbol
        Rank
        Durations
        GrowthHabits
        NativeStatuses

    Do not recursively traverse Ancestors.
    """
    if not isinstance(payload, dict):
        raise TypeError(
            f"{symbol}: expected dict response, got {type(payload).__name__}"
        )

    returned_symbol = clean_text(payload.get("Symbol"))

    if pd.notna(returned_symbol) and str(returned_symbol).strip() != symbol:
        raise ValueError(
            f"Requested {symbol}, USDA returned {returned_symbol}"
        )

    duration = unique_pipe(payload.get("Durations"))
    growth_habit = unique_pipe(payload.get("GrowthHabits"))

    l48_status = pd.NA
    l48_type = pd.NA

    for item in (payload.get("NativeStatuses") or []):
        if not isinstance(item, dict):
            continue

        region = clean_text(item.get("Region"))

        if pd.notna(region) and str(region).strip() == "L48":
            raw_status = clean_text(item.get("Status"))
            raw_type = clean_text(item.get("Type"))

            # Prefer Status, but Type is a valid fallback.
            l48_status = canonical_l48_status(
                raw_status if pd.notna(raw_status) else raw_type
            )

            l48_type = (
                raw_type
                if pd.notna(raw_type)
                else canonical_l48_type(l48_status)
            )
            break

    retrieved_utc = datetime.now(timezone.utc).isoformat()

    return {
        "USDA_accepted_symbol": symbol,
        "USDA_plant_id": payload.get("Id"),
        "USDA_rank": clean_text(payload.get("Rank")),
        "USDA_duration": duration,
        "USDA_growth_habit": growth_habit,
        "USDA_native_status_L48": l48_status,
        "USDA_native_type_L48": l48_type,
        "USDA_profile_verified_root_parser": True,
        "USDA_profile_retrieved_utc": retrieved_utc,
        "USDA_profile_parser_version": "root_PlantProfile_v1",
    }


def fetch_usda_profile(symbol):
    url = f"{USDA_SERVICE_BASE}/PlantProfile"
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(
                url,
                params={"symbol": symbol},
                timeout=REQUEST_TIMEOUT,
                headers={
                    "Accept": "application/json",
                    "User-Agent": "MOSAIC-USDA-trait-cache/2.0",
                },
            )
            response.raise_for_status()

            return parse_profile_payload(
                symbol,
                response.json(),
            )

        except Exception as exc:
            last_error = exc

        if attempt < MAX_RETRIES:
            time.sleep(
                REQUEST_DELAY_SECONDS * (2 ** (attempt - 1))
            )

    raise RuntimeError(
        f"{symbol}: {repr(last_error)}"
    )


def coalesce_columns(df, target, aliases):
    """
    Coalesce historical aliases into one canonical column.
    Existing canonical values take precedence.
    """
    if target not in df.columns:
        df[target] = pd.Series(pd.NA, index=df.index, dtype="string")

    df[target] = df[target].astype("string")

    for alias in aliases:
        if alias in df.columns and alias != target:
            candidate = df[alias].astype("string")
            df[target] = df[target].combine_first(candidate)

    return df


# ----------------------------------------------------------------------------
# Required USDA accepted-symbol universe
# ----------------------------------------------------------------------------

required_symbols = sorted(
    set(
        dictionary.loc[
            dictionary["code_class"].eq("Plant")
            & dictionary["USDA_accepted_symbol"].notna(),
            "USDA_accepted_symbol",
        ]
        .astype(str)
        .str.strip()
        .str.upper()
    )
)

print(f"Required USDA accepted symbols: {len(required_symbols):,}")


# ----------------------------------------------------------------------------
# Load or initialize cache
# ----------------------------------------------------------------------------

if USDA_ATTRIBUTE_CACHE.exists():

    cache = pd.read_csv(
        USDA_ATTRIBUTE_CACHE,
        dtype="string",
        low_memory=False,
    ).copy()

else:

    cache = pd.DataFrame(
        columns=[
            "USDA_accepted_symbol",
            "USDA_plant_id",
            "USDA_rank",
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
            "USDA_native_type_L48",
            "USDA_profile_verified_root_parser",
            "USDA_profile_retrieved_utc",
            "USDA_profile_parser_version",
        ]
    )

print(f"Existing cache rows: {len(cache):,}")


# ----------------------------------------------------------------------------
# Normalize historical cache schemas
# ----------------------------------------------------------------------------

cache = coalesce_columns(
    cache,
    "USDA_accepted_symbol",
    [
        "accepted_symbol",
        "Symbol",
        "USDA_symbol",
    ],
)

cache = coalesce_columns(
    cache,
    "USDA_duration",
    [
        "USDA_duration_api",
        "duration",
        "Duration",
    ],
)

cache = coalesce_columns(
    cache,
    "USDA_growth_habit",
    [
        "USDA_growth_habit_api",
        "growth_habit",
        "GrowthHabit",
        "Growth Habit",
    ],
)

cache = coalesce_columns(
    cache,
    "USDA_native_status_L48",
    [
        "USDA_L48_native_status_api",
        "native_status_l48",
        "NativeStatus_L48",
        "L48_native_status",
        "L48_status",
    ],
)

cache = coalesce_columns(
    cache,
    "USDA_native_type_L48",
    [
        "USDA_L48_native_type_api",
        "native_type_l48",
        "NativeType_L48",
    ],
)

for col in [
    "USDA_profile_verified_root_parser",
    "USDA_profile_retrieved_utc",
    "USDA_profile_parser_version",
    "USDA_plant_id",
    "USDA_rank",
]:
    if col not in cache.columns:
        cache[col] = pd.NA

# Clean canonical fields.
for col in [
    "USDA_accepted_symbol",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "USDA_native_type_L48",
]:
    cache[col] = (
        cache[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        })
    )

cache["USDA_accepted_symbol"] = (
    cache["USDA_accepted_symbol"]
    .str.upper()
)

cache["USDA_native_status_L48"] = (
    cache["USDA_native_status_L48"]
    .apply(canonical_l48_status)
    .astype("string")
)

cache["USDA_native_type_L48"] = pd.Series(
    [
        (
            clean_text(t)
            if pd.notna(clean_text(t))
            else canonical_l48_type(s)
        )
        for t, s in zip(
            cache["USDA_native_type_L48"],
            cache["USDA_native_status_L48"],
        )
    ],
    index=cache.index,
    dtype="string",
)

cache["USDA_profile_verified_root_parser"] = (
    cache["USDA_profile_verified_root_parser"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

# One row per accepted symbol before audit.
dups = cache["USDA_accepted_symbol"].duplicated(keep=False)

if dups.any():
    # Prefer a root-verified row; otherwise retain the last available row.
    cache["_verified_sort"] = (
        cache["USDA_profile_verified_root_parser"]
        .fillna(False)
        .astype(int)
    )

    cache = (
        cache.sort_values(
            [
                "USDA_accepted_symbol",
                "_verified_sort",
            ]
        )
        .drop_duplicates(
            subset=["USDA_accepted_symbol"],
            keep="last",
        )
        .drop(columns="_verified_sort")
        .reset_index(drop=True)
    )

    print(
        "Collapsed historical duplicate cache symbols to one row each."
    )


# ----------------------------------------------------------------------------
# Determine recovery targets
# ----------------------------------------------------------------------------

cache_idx = cache.set_index(
    "USDA_accepted_symbol",
    drop=False,
)

absent_symbols = []

incomplete_unverified_symbols = []

for symbol in required_symbols:

    if symbol not in cache_idx.index:
        absent_symbols.append(symbol)
        continue

    row = cache_idx.loc[symbol]

    # Defensive: after deduplication this should be a Series.
    if isinstance(row, pd.DataFrame):
        row = row.iloc[-1]

    incomplete = any(
        pd.isna(row.get(col))
        for col in [
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
        ]
    )

    verified = bool(
        row.get(
            "USDA_profile_verified_root_parser",
            False,
        )
    )

    if incomplete and not verified:
        incomplete_unverified_symbols.append(symbol)


recovery_targets = sorted(
    set(
        absent_symbols
        + incomplete_unverified_symbols
    )
)

print()
print("USDA CACHE AUDIT")
print("-" * 70)
print(f"Required symbols:                 {len(required_symbols):,}")
print(f"Absent from cache:                {len(absent_symbols):,}")
print(
    "Incomplete + not root-verified:  "
    f"{len(incomplete_unverified_symbols):,}"
)
print(f"Profiles to query this run:       {len(recovery_targets):,}")


# ----------------------------------------------------------------------------
# Query only required recovery targets
# ----------------------------------------------------------------------------

recovered_rows = []
failure_rows = []

for i, symbol in enumerate(
    recovery_targets,
    start=1,
):

    try:

        row = fetch_usda_profile(symbol)
        recovered_rows.append(row)

        print(
            f"[{i:04d}/{len(recovery_targets):04d}] "
            f"{symbol:<12} "
            f"D={str(row['USDA_duration']):<24} "
            f"H={str(row['USDA_growth_habit']):<28} "
            f"L48={str(row['USDA_native_status_L48'])}"
        )

    except Exception as exc:

        failure_rows.append({
            "USDA_accepted_symbol": symbol,
            "error": repr(exc),
            "failed_utc": datetime.now(timezone.utc).isoformat(),
        })

        print(
            f"[{i:04d}/{len(recovery_targets):04d}] "
            f"{symbol:<12} FAILED | {repr(exc)}"
        )

    time.sleep(REQUEST_DELAY_SECONDS)


# ----------------------------------------------------------------------------
# Update cache only when recoveries succeeded
# ----------------------------------------------------------------------------

if recovered_rows:

    recovered = pd.DataFrame(recovered_rows)

    recovered_symbols = set(
        recovered["USDA_accepted_symbol"]
    )

    cache = cache.loc[
        ~cache["USDA_accepted_symbol"].isin(
            recovered_symbols
        )
    ].copy()

    cache = pd.concat(
        [
            cache,
            recovered,
        ],
        ignore_index=True,
        sort=False,
    )

    cache = (
        cache.sort_values("USDA_accepted_symbol")
        .reset_index(drop=True)
    )

    # Timestamped backup only when an existing cache will be changed.
    if USDA_ATTRIBUTE_CACHE.exists():

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        backup = (
            OUTPUT_DIR
            / f"USDA_PLANTS_ecological_attributes_backup_{stamp}.csv"
        )

        shutil.copy2(
            USDA_ATTRIBUTE_CACHE,
            backup,
        )

        print("Cache backup:", backup)

    cache.to_csv(
        USDA_ATTRIBUTE_CACHE,
        index=False,
    )

    print(
        f"Updated authoritative cache written: "
        f"{len(cache):,} rows"
    )

elif not USDA_ATTRIBUTE_CACHE.exists():

    # Create an empty-but-canonical cache if every initial query failed.
    cache.to_csv(
        USDA_ATTRIBUTE_CACHE,
        index=False,
    )


# ----------------------------------------------------------------------------
# Persist failures separately; these remain retryable next run.
# ----------------------------------------------------------------------------

if failure_rows:

    failures = pd.DataFrame(failure_rows)

    if USDA_RECOVERY_FAILURE_FILE.exists():

        prior = pd.read_csv(
            USDA_RECOVERY_FAILURE_FILE,
            dtype="string",
            low_memory=False,
        )

        failures = pd.concat(
            [prior, failures],
            ignore_index=True,
            sort=False,
        )

    failures = (
        failures
        .drop_duplicates(
            subset=[
                "USDA_accepted_symbol",
                "error",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    failures.to_csv(
        USDA_RECOVERY_FAILURE_FILE,
        index=False,
    )

    print(
        f"Recovery failures this run: {len(failure_rows):,}"
    )


# ----------------------------------------------------------------------------
# Reload canonical cache after update
# ----------------------------------------------------------------------------

cache = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()

cache["USDA_accepted_symbol"] = (
    cache["USDA_accepted_symbol"]
    .astype("string")
    .str.strip()
    .str.upper()
)

cached_symbols = set(
    cache["USDA_accepted_symbol"]
    .dropna()
)

still_absent = sorted(
    set(required_symbols)
    - cached_symbols
)

print()
print(
    f"Required USDA symbols still absent after recovery: "
    f"{len(still_absent):,}"
)

if still_absent:
    print(
        "These taxa remain usable as known taxa but their ecological "
        "traits will be unavailable until a future successful query."
    )
    print(still_absent[:50])


Required USDA accepted symbols: 7,268
Existing cache rows: 7,269

USDA CACHE AUDIT
----------------------------------------------------------------------
Required symbols:                 7,268
Absent from cache:                0
Incomplete + not root-verified:  0
Profiles to query this run:       0

Required USDA symbols still absent after recovery: 0


In [108]:
# ============================================================================
# 17b. CANONICAL STRUCTURAL FUNCTIONAL-GROUP RULES
# ============================================================================

def pipe_tokens(value):
    """Parse pipe-delimited USDA fields without modifying the source value."""
    if pd.isna(value):
        return set()

    return {
        x.strip().lower()
        for x in str(value).split("|")
        if x.strip()
    }


def structural_from_usda(growth_value, duration_value):
    """
    Derive MOSAIC structural / life-history functional group from USDA traits.

    Nativity is intentionally NOT used here.

    Key rules:
      Lichenous                    -> Lichen
      Nonvascular                 -> Nonvascular

      Graminoid + Annual          -> AnnualGrass
      Graminoid + Perennial       -> PerennialGrass
      Graminoid + ambiguous/NA    -> Grass

      Forb/herb + Annual          -> AnnualForb
      Forb/herb + Annual|Biennial -> AnnualForb
      Forb/herb + Biennial        -> BiennialForb
      Forb/herb + Perennial       -> PerennialForb
      Forb/herb + ambiguous/NA    -> Forb

      Shrub|Tree and similar mixed woody forms -> Woody
    """

    growth = pipe_tokens(growth_value)
    duration = pipe_tokens(duration_value)

    if not growth:
        return pd.NA

    annual = "annual" in duration
    biennial = "biennial" in duration
    perennial = "perennial" in duration

    # ------------------------------------------------------------------------
    # Lichens / nonvascular
    # ------------------------------------------------------------------------

    if growth == {"lichenous"}:
        return "Lichen"

    if growth == {"nonvascular"}:
        return "Nonvascular"

    # ------------------------------------------------------------------------
    # Graminoids
    # ------------------------------------------------------------------------

    if growth == {"graminoid"}:

        if annual and not perennial:
            return "AnnualGrass"

        if perennial and not annual:
            return "PerennialGrass"

        # Annual|Perennial or missing/other duration remains broad.
        return "Grass"

    # ------------------------------------------------------------------------
    # Herbaceous forbs
    #
    # Forb/herb|Vine is treated as forb structurally.
    # ------------------------------------------------------------------------

    if "forb/herb" in growth:

        nonherb = growth - {
            "forb/herb",
            "forb",
            "herb",
            "vine",
        }

        if not nonherb:

            # Annual and Annual|Biennial both remain annual-life-history forbs.
            if annual and not perennial:
                return "AnnualForb"

            # Pure biennial receives its own structural class.
            if (
                biennial
                and not annual
                and not perennial
            ):
                return "BiennialForb"

            if perennial and not annual:
                return "PerennialForb"

            # Annual|Perennial, Annual|Biennial|Perennial,
            # missing duration, etc.
            return "Forb"

    # ------------------------------------------------------------------------
    # Pure woody habits
    # ------------------------------------------------------------------------

    if growth == {"shrub"}:
        return "Shrub"

    if growth == {"subshrub"}:
        return "Subshrub"

    if growth == {"tree"}:
        return "Tree"

    # Shrub + subshrub remains shrub-domain.
    if (
        growth
        and growth.issubset(
            {
                "shrub",
                "subshrub",
            }
        )
    ):
        return "Shrub"

    # Mixed woody categories retain Woody.
    woody = {
        "shrub",
        "subshrub",
        "tree",
    }

    if (
        len(growth.intersection(woody)) >= 2
        and growth.issubset(woody)
    ):
        return "Woody"

    # Herbaceous + woody combinations are genuinely mixed woody forms.
    if (
        "forb/herb" in growth
        and growth.intersection(woody)
    ):
        return "Woody"

    # ------------------------------------------------------------------------
    # Vine-only and genuinely mixed USDA habits
    # ------------------------------------------------------------------------

    if growth == {"vine"}:
        return "Vine"

    return "MixedGrowthHabit"


# Quick rule checks.
assert structural_from_usda("Lichenous", pd.NA) == "Lichen"
assert structural_from_usda("Forb/herb", "Biennial") == "BiennialForb"
assert structural_from_usda("Forb/herb", "Annual|Biennial") == "AnnualForb"
assert structural_from_usda("Forb/herb", "Perennial") == "PerennialForb"
assert structural_from_usda("Forb/herb", "Annual|Perennial") == "Forb"

print("Canonical structural FG rules loaded.")

Canonical structural FG rules loaded.


## 18. Merge the completed USDA ecological-trait cache

At this point the cache has already been normalized, audited against the exact accepted-symbol universe required by the dictionary, and repaired where possible. This cell performs one deterministic merge.

A USDA-resolved taxon may still have missing traits if the current USDA root profile itself lacks them or if retrieval failed. That is allowed: functional-group resolution degrades to the most informative broad class available, and universal hit interpretation remains non-null.


In [109]:
# ============================================================================
# 18. MERGE COMPLETED USDA ECOLOGICAL-TRAIT CACHE
# ============================================================================

traits = pd.read_csv(
    USDA_ATTRIBUTE_CACHE,
    dtype="string",
    low_memory=False,
).copy()

print(f"Trait-cache rows: {len(traits):,}")

# Canonical columns should now exist because the preceding audit created them.
required_trait_cols = [
    "USDA_accepted_symbol",
    "USDA_duration",
    "USDA_growth_habit",
    "USDA_native_status_L48",
    "USDA_native_type_L48",
]

missing_required = [
    c for c in required_trait_cols
    if c not in traits.columns
]

assert not missing_required, (
    f"Trait cache is missing canonical columns: {missing_required}"
)

for col in required_trait_cols:
    traits[col] = (
        traits[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        })
    )

traits["USDA_accepted_symbol"] = (
    traits["USDA_accepted_symbol"]
    .str.upper()
)

traits["USDA_native_status_L48"] = (
    traits["USDA_native_status_L48"]
    .apply(canonical_l48_status)
    .astype("string")
)

traits["USDA_native_type_L48"] = pd.Series(
    [
        (
            clean_text(t)
            if pd.notna(clean_text(t))
            else canonical_l48_type(s)
        )
        for t, s in zip(
            traits["USDA_native_type_L48"],
            traits["USDA_native_status_L48"],
        )
    ],
    index=traits.index,
    dtype="string",
)

dup = traits["USDA_accepted_symbol"].duplicated(keep=False)

assert not dup.any(), (
    "Authoritative ecological-trait cache contains duplicate accepted symbols."
)

trait_cols = list(required_trait_cols)

for c in [
    "USDA_plant_id",
    "USDA_rank",
    "USDA_profile_verified_root_parser",
    "USDA_profile_retrieved_utc",
    "USDA_profile_parser_version",
]:
    if c in traits.columns:
        trait_cols.append(c)

dictionary_fg = dictionary.copy()

# Remove stale ecological columns if the notebook is rerun interactively.
stale_trait_cols = [
    c for c in trait_cols
    if c != "USDA_accepted_symbol"
    and c in dictionary_fg.columns
]

dictionary_fg = dictionary_fg.drop(
    columns=stale_trait_cols,
    errors="ignore",
)

dictionary_fg = dictionary_fg.merge(
    traits[trait_cols],
    how="left",
    on="USDA_accepted_symbol",
    validate="many_to_one",
)

dictionary_fg["USDA_symbol_in_trait_cache"] = (
    dictionary_fg["USDA_accepted_symbol"].notna()
    & dictionary_fg["USDA_accepted_symbol"].isin(
        set(
            traits["USDA_accepted_symbol"]
            .dropna()
        )
    )
)

print(
    "USDA-resolved plant codes:",
    f"{(
        dictionary_fg['code_class'].eq('Plant')
        & dictionary_fg['USDA_accepted_symbol'].notna()
    ).sum():,}"
)

print(
    "USDA-resolved plant codes present in trait cache:",
    f"{(
        dictionary_fg['code_class'].eq('Plant')
        & dictionary_fg['USDA_accepted_symbol'].notna()
        & dictionary_fg['USDA_symbol_in_trait_cache']
    ).sum():,}"
)


Trait-cache rows: 7,269
USDA-resolved plant codes: 7,605
USDA-resolved plant codes present in trait cache: 7,605


## 19. Canonical two-level MOSAIC functional-group classification

The notebook now separates two ecological questions:

1. **`MOSAIC_FG_structural`** — growth form + life history. Nativity is not required.
2. **`MOSAIC_FG`** — the refined class. Nativity is used when available; otherwise the structural class is retained.

Examples:

- Annual + Graminoid + Introduced → `AnnualGrass` / `EAG`
- Annual + Graminoid + missing nativity → `AnnualGrass` / `AnnualGrass`
- Perennial + Graminoid + Native → `PerennialGrass` / `NPG`
- Annual + Forb/herb + Introduced → `AnnualForb` / `ExoticAnnualForb`
- Perennial + Forb/herb + Native → `PerennialForb` / `NativePerennialForb`
- `Shrub|Tree` or `Shrub|Subshrub|Tree` → `Woody`
- Source/project classifications retain precedence.


In [110]:
# ============================================================================
# TWO-LEVEL MOSAIC FUNCTIONAL-GROUP CLASSIFICATION
# ============================================================================

def pipe_tokens(value):
    if pd.isna(value):
        return set()
    return {
        x.strip().lower()
        for x in str(value).split("|")
        if x.strip()
    }


def structural_from_usda(growth_value, duration_value):
    growth = pipe_tokens(growth_value)
    duration = pipe_tokens(duration_value)

    if not growth:
        return pd.NA

    annual = "annual" in duration
    perennial = "perennial" in duration

    # Nonvascular
    if growth == {"nonvascular"}:
        return "Nonvascular"

    # Graminoids
    if growth == {"graminoid"}:
        if annual and not perennial:
            return "AnnualGrass"
        if perennial and not annual:
            return "PerennialGrass"
        return "Grass"

    # Herbaceous forb; vine is treated as a secondary habit.
    if "forb/herb" in growth:
        nonherb = growth - {"forb/herb", "forb", "herb", "vine"}
        if not nonherb:
            if annual and not perennial:
                return "AnnualForb"
            if perennial and not annual:
                return "PerennialForb"
            return "Forb"

    # Pure woody categories.
    if growth == {"shrub"}:
        return "Shrub"
    if growth == {"subshrub"}:
        return "Subshrub"
    if growth == {"tree"}:
        return "Tree"

    # Shrub + subshrub stays shrub-domain.
    if growth and growth.issubset({"shrub", "subshrub"}):
        return "Shrub"

    # Mixed shrub/subshrub/tree combinations retain Woody.
    woody = {"shrub", "subshrub", "tree"}
    if (
        len(growth.intersection(woody)) >= 2
        and growth.issubset(woody)
    ):
        return "Woody"

    # Herbaceous + woody combinations are genuinely mixed.
    if "forb/herb" in growth and growth.intersection(woody):
        return "Woody"

    if growth == {"vine"}:
        return "Vine"

    # Keep complex USDA mixtures explicit rather than forcing a dominant form.
    return "MixedGrowthHabit"


dictionary_fg["USDA_derived_group"] = pd.Series(
    [
        structural_from_usda(g, d)
        for g, d in zip(
            dictionary_fg["USDA_growth_habit"],
            dictionary_fg["USDA_duration"],
        )
    ],
    index=dictionary_fg.index,
    dtype="string",
)

# ----------------------------------------------------------------------------
# Map explicit source/project classes into the structural vocabulary.
# These remain higher-priority than USDA fallback.
# ----------------------------------------------------------------------------

def source_structural(group):
    if pd.isna(group):
        return pd.NA

    g = str(group).strip()

    lookup = {
        "PBG": "PBG",
        "PerennialBunchgrass": "PBG",
        "Perennial bunchgrass": "PBG",

        "AnnualGraminoid": "AnnualGrass",
        "Annual graminoid": "AnnualGrass",
        "PerennialGraminoid": "PerennialGrass",
        "Perennial graminoid": "PerennialGrass",
        "Graminoid": "Grass",
        "Unknown grass": "Grass",

        "AnnualForb": "AnnualForb",
        "Annual forb": "AnnualForb",

        "BiennialForb": "BiennialForb",
        "Biennial forb": "BiennialForb",

        "PerennialForb": "PerennialForb",
        "Perennial forb": "PerennialForb",

        "Forb": "Forb",

        "Shrub": "Shrub",
        "SHRUB": "Shrub",
        "Unknown shrub": "Shrub",
        "Subshrub": "Subshrub",
        "Tree": "Tree",

        "Lichen": "Lichen",
        "Nonvascular": "Nonvascular",

        "UnknownPlant": "UnknownPlant",
        "Unknown plant": "UnknownPlant",
    }

    return lookup.get(g, g)


source_struct = (
    dictionary_fg["protocol_plant_group"]
    .apply(source_structural)
    .astype("string")
)

# Structural label: source/project information first, USDA fallback second.
dictionary_fg["MOSAIC_FG_structural"] = (
    source_struct
    .combine_first(dictionary_fg["USDA_derived_group"])
    .astype("string")
)

# ----------------------------------------------------------------------------
# Nativity-refined label.
# ----------------------------------------------------------------------------

def refine_fg(structural, status):
    if pd.isna(structural):
        return pd.NA

    s = str(structural)

    nat = (
        set()
        if pd.isna(status)
        else pipe_tokens(status)
    )

    introduced = "i" in nat
    native = "n" in nat

    # ------------------------------------------------------------------------
    # GRASSES
    # ------------------------------------------------------------------------

    if s == "AnnualGrass":
        if introduced:
            return "EAG"
        if native:
            return "NAG"
        return "AnnualGrass"

    if s == "PerennialGrass":
        if introduced:
            return "EPG"
        if native:
            return "NPG"
        return "PerennialGrass"

    # ------------------------------------------------------------------------
    # FORBS
    # ------------------------------------------------------------------------

    if s == "AnnualForb":
        if introduced:
            return "ExoticAnnualForb"
        if native:
            return "NativeAnnualForb"
        return "AnnualForb"

    if s == "BiennialForb":
        if introduced:
            return "ExoticBiennialForb"
        if native:
            return "NativeBiennialForb"

        # Nativity unresolved, but life history and growth habit are known.
        return "BiennialForb"

    if s == "PerennialForb":
        if introduced:
            return "ExoticPerennialForb"
        if native:
            return "NativePerennialForb"
        return "PerennialForb"

    if s == "Forb":
        if introduced:
            return "ExoticForb"
        if native:
            return "NativeForb"
        return "Forb"

    # ------------------------------------------------------------------------
    # EVERYTHING ELSE
    # ------------------------------------------------------------------------

    return s


dictionary_fg["USDA_MOSAIC_FG"] = pd.Series(
    [
        refine_fg(s, n)
        for s, n in zip(
            dictionary_fg["USDA_derived_group"],
            dictionary_fg["USDA_native_status_L48"],
        )
    ],
    index=dictionary_fg.index,
    dtype="string",
)

# Final refined FG uses source structural group first, then nativity refinement.
dictionary_fg["MOSAIC_FG"] = pd.Series(
    [
        refine_fg(s, n)
        for s, n in zip(
            dictionary_fg["MOSAIC_FG_structural"],
            dictionary_fg["USDA_native_status_L48"],
        )
    ],
    index=dictionary_fg.index,
    dtype="string",
)

# Unknown USDA-resolved plants with no usable structure stay explicitly unresolved.
plant = dictionary_fg["code_class"].eq("Plant")
known_taxon = dictionary_fg["USDA_accepted_symbol"].notna()

dictionary_fg.loc[
    plant & known_taxon & dictionary_fg["MOSAIC_FG"].isna(),
    "MOSAIC_FG",
] = "TraitUnresolvedPlant"

dictionary_fg.loc[
    plant & ~known_taxon & dictionary_fg["MOSAIC_FG"].isna(),
    "MOSAIC_FG",
] = "UnknownPlant"

# Compatible legacy field retained for downstream code.
dictionary_fg["resolved_plant_group"] = (
    dictionary_fg["MOSAIC_FG_structural"]
)

# ----------------------------------------------------------------------------
# Functional-group provenance.
# ----------------------------------------------------------------------------

dictionary_fg["FG_resolution_source"] = pd.NA

source_mask = dictionary_fg["protocol_plant_group"].notna()
dictionary_fg.loc[
    source_mask,
    "FG_resolution_source",
] = "protocol_or_project_group"

dictionary_fg.loc[
    dictionary_fg["protocol_plant_group"].isin(
        ["PBG", "PerennialBunchgrass", "Perennial bunchgrass"]
    ),
    "FG_resolution_source",
] = "explicit_PBG_source"

dictionary_fg.loc[
    ~source_mask
    & dictionary_fg["USDA_derived_group"].notna(),
    "FG_resolution_source",
] = "USDA_two_level_traits"

dictionary_fg.loc[
    plant
    & known_taxon
    & dictionary_fg["USDA_derived_group"].isna(),
    "FG_resolution_source",
] = "USDA_taxon_missing_usable_structure"

dictionary_fg.loc[
    plant
    & ~known_taxon
    & dictionary_fg["FG_resolution_source"].isna(),
    "FG_resolution_source",
] = "plant_without_USDA_taxon"

# ----------------------------------------------------------------------------
# Trait completeness is diagnostic only; it no longer determines whether FG
# resolution succeeds.
# ----------------------------------------------------------------------------

def trait_status(row):
    if pd.isna(row.get("code_class")) or row.get("code_class") != "Plant":
        return pd.NA

    if pd.isna(row.get("USDA_accepted_symbol")):
        return "No_USDA_taxon"

    missing = []
    if pd.isna(row.get("USDA_duration")):
        missing.append("duration")
    if pd.isna(row.get("USDA_growth_habit")):
        missing.append("growth_habit")
    if pd.isna(row.get("USDA_native_status_L48")):
        missing.append("nativity")

    if missing:
        return "Missing_" + "_".join(missing)

    return "Traits_complete"


dictionary_fg["USDA_trait_status"] = dictionary_fg.apply(
    trait_status,
    axis=1,
)

# ----------------------------------------------------------------------------
# Known manual ecological override: Bassia variants.
# Preserve accepted taxonomy while overriding ecological FG interpretation.
# ----------------------------------------------------------------------------

bassia = dictionary_fg["observed_code"].isin(["BAPRV", "BAPRG"])

dictionary_fg.loc[bassia, "MOSAIC_FG_structural"] = "Forb"
dictionary_fg.loc[bassia, "resolved_plant_group"] = "Forb"
dictionary_fg.loc[bassia, "MOSAIC_FG"] = "ExoticForb"
dictionary_fg.loc[
    bassia,
    "FG_resolution_source",
] = "manual_ecological_functional_override"

print("Two-level functional-group classification complete.")


Two-level functional-group classification complete.


## 20. Build a universal interpretation for every observed LPI code

The functional-group fields are plant-focused, but the final dictionary must be able to interpret **every** hit in the 16-million-record LPI table.

This section therefore adds explicit lookup/provenance fields:

- `canonical_code` — accepted USDA symbol when available; otherwise the immutable observed code.
- `canonical_label` — USDA scientific name when available; otherwise the resolved protocol/project label; otherwise an explicit unknown label containing the source code.
- `dictionary_resolution_status` — tells downstream code whether the row is an exact USDA taxon, corrected USDA taxon, known non-USDA plant, unknown plant, no-canopy observation, non-plant class, or fully unknown code.
- `hit_interpretation` — guaranteed non-null interpretation for joining to every LPI hit.

Oddballs that cannot yet be repaired (for example codes analogous to `GUTcfSAR`) remain in the dictionary under their exact observed code and resolve explicitly to an unknown interpretation rather than being dropped.


In [111]:
# ============================================================================
# 20. UNIVERSAL HIT INTERPRETATION
# ============================================================================

# Ensure every row has an explicit broad code class.
dictionary_fg["code_class"] = (
    dictionary_fg["code_class"]
    .astype("string")
)

dictionary_fg.loc[
    dictionary_fg["code_class"].isna(),
    "code_class",
] = "Unknown"


# ============================================================================
# 1. CANONICAL CODE
#
# Prefer accepted USDA symbol when available.
# Otherwise preserve exact observed source code.
# ============================================================================

dictionary_fg["canonical_code"] = (
    dictionary_fg["USDA_accepted_symbol"]
    .astype("string")
    .combine_first(
        dictionary_fg["observed_code"]
        .astype("string")
    )
)


# ============================================================================
# 2. CANONICAL LABEL
#
# Prefer USDA scientific name.
# Then use resolved protocol/project label.
# Finally fall back to an explicit unknown label.
# ============================================================================

dictionary_fg["canonical_label"] = (
    dictionary_fg["USDA_scientific_name"]
    .astype("string")
    .combine_first(
        dictionary_fg["resolved_label"]
        .astype("string")
    )
)

unknown_label_mask = (
    dictionary_fg["canonical_label"]
    .isna()
)

dictionary_fg.loc[
    unknown_label_mask,
    "canonical_label",
] = (
    "Unknown code: "
    + dictionary_fg.loc[
        unknown_label_mask,
        "observed_code",
    ].astype("string")
)


# ============================================================================
# 3. BASE BOOLEAN MASKS
#
# Force nullable comparisons to pure boolean before combining masks.
# ============================================================================

is_plant = (
    dictionary_fg["code_class"]
    .eq("Plant")
    .fillna(False)
)

is_no_canopy = (
    dictionary_fg["code_class"]
    .eq("NoCanopy")
    .fillna(False)
)

is_unknown_class = (
    dictionary_fg["code_class"]
    .eq("Unknown")
    .fillna(False)
)

has_usda = (
    dictionary_fg["USDA_accepted_symbol"]
    .notna()
)

same_as_accepted = (
    dictionary_fg["observed_code"]
    .astype("string")
    .eq(
        dictionary_fg["USDA_accepted_symbol"]
        .astype("string")
    )
    .fillna(False)
)


# ============================================================================
# 4. DICTIONARY RESOLUTION STATUS
# ============================================================================

dictionary_fg["dictionary_resolution_status"] = pd.Series(
    pd.NA,
    index=dictionary_fg.index,
    dtype="string",
)


# --------------------------------------------------------------------------
# Exact USDA taxon
#
# Example:
# observed_code = BRTE
# USDA_accepted_symbol = BRTE
# --------------------------------------------------------------------------

exact_usda = (
    is_plant
    & has_usda
    & same_as_accepted
)

dictionary_fg.loc[
    exact_usda,
    "dictionary_resolution_status",
] = "USDA_exact_taxon"


# --------------------------------------------------------------------------
# Corrected or synonymized USDA taxon
#
# Example:
# observed_code = POAR2R2
# USDA_accepted_symbol = PONIN
# --------------------------------------------------------------------------

corrected_usda = (
    is_plant
    & has_usda
    & ~same_as_accepted
)

dictionary_fg.loc[
    corrected_usda,
    "dictionary_resolution_status",
] = "USDA_corrected_or_synonym_taxon"


# --------------------------------------------------------------------------
# Non-USDA plant that is still functionally resolved
#
# These are protocol/project/local vegetation codes where we cannot assign
# a USDA accepted taxon, but we still have enough information to assign
# a meaningful MOSAIC functional group.
# --------------------------------------------------------------------------

known_non_usda_plant = (
    is_plant
    & ~has_usda
    & dictionary_fg["MOSAIC_FG"].notna()
    & ~(
        dictionary_fg["MOSAIC_FG"]
        .eq("UnknownPlant")
        .fillna(False)
    )
)

dictionary_fg.loc[
    known_non_usda_plant,
    "dictionary_resolution_status",
] = "NonUSDA_plant_functionally_resolved"


# --------------------------------------------------------------------------
# Remaining plant codes are explicitly UnknownPlant
#
# We know the source code represents vegetation, but cannot resolve it to
# taxon or functional group more specifically.
# --------------------------------------------------------------------------

unknown_plant = (
    is_plant
    & dictionary_fg[
        "dictionary_resolution_status"
    ].isna()
)

dictionary_fg.loc[
    unknown_plant,
    "dictionary_resolution_status",
] = "UnknownPlant"


# --------------------------------------------------------------------------
# No-canopy
#
# Protocol state, not an unknown.
# --------------------------------------------------------------------------

dictionary_fg.loc[
    is_no_canopy,
    "dictionary_resolution_status",
] = "NoCanopy"


# --------------------------------------------------------------------------
# Other classified non-plant entities
#
# These may include soil-surface classes, litter, rock, bare ground, etc.
# --------------------------------------------------------------------------

classified_other = (
    ~is_plant
    & ~is_no_canopy
    & ~is_unknown_class
    & dictionary_fg[
        "dictionary_resolution_status"
    ].isna()
)

dictionary_fg.loc[
    classified_other,
    "dictionary_resolution_status",
] = "NonPlant_classified"


# --------------------------------------------------------------------------
# Truly unresolvable source codes
#
# These are the GUTcfSAR-type oddballs:
# we cannot support a biological or protocol interpretation.
# Preserve the exact observed code, but classify the interpretation as Unknown.
# --------------------------------------------------------------------------

dictionary_fg.loc[
    dictionary_fg[
        "dictionary_resolution_status"
    ].isna(),
    "dictionary_resolution_status",
] = "UnknownCode"


# ============================================================================
# 5. PLANT FUNCTIONAL-GROUP FALLBACKS
#
# Every plant should retain at least UnknownPlant.
# ============================================================================

plant_missing_structural = (
    is_plant
    & dictionary_fg[
        "MOSAIC_FG_structural"
    ].isna()
)

dictionary_fg.loc[
    plant_missing_structural,
    "MOSAIC_FG_structural",
] = "UnknownPlant"


plant_missing_refined = (
    is_plant
    & dictionary_fg[
        "MOSAIC_FG"
    ].isna()
)

dictionary_fg.loc[
    plant_missing_refined,
    "MOSAIC_FG",
] = "UnknownPlant"


# ============================================================================
# 6. UNIVERSAL HIT INTERPRETATION
#
# Every row must receive a non-null interpretation.
# ============================================================================

dictionary_fg["hit_interpretation"] = pd.Series(
    pd.NA,
    index=dictionary_fg.index,
    dtype="string",
)


# --------------------------------------------------------------------------
# Plant rows use best available MOSAIC FG.
# --------------------------------------------------------------------------

dictionary_fg.loc[
    is_plant,
    "hit_interpretation",
] = (
    dictionary_fg.loc[
        is_plant,
        "MOSAIC_FG",
    ]
    .astype("string")
)


# --------------------------------------------------------------------------
# No-canopy
# --------------------------------------------------------------------------

dictionary_fg.loc[
    is_no_canopy,
    "hit_interpretation",
] = "NO_CANOPY"


# --------------------------------------------------------------------------
# Other classified non-plants use their code class.
# --------------------------------------------------------------------------

other_nonplant = (
    ~is_plant
    & ~is_no_canopy
    & ~is_unknown_class
)

dictionary_fg.loc[
    other_nonplant,
    "hit_interpretation",
] = (
    dictionary_fg.loc[
        other_nonplant,
        "code_class",
    ]
    .astype("string")
)


# --------------------------------------------------------------------------
# Final generic fallback
# --------------------------------------------------------------------------

dictionary_fg.loc[
    dictionary_fg[
        "hit_interpretation"
    ].isna(),
    "hit_interpretation",
] = "Unknown"


# ============================================================================
# 7. FORCE TRULY UNRESOLVABLE CODES TO UNKNOWN
#
# UnknownCode means:
#   - source code is retained exactly
#   - no structural FG is asserted
#   - no refined FG is asserted
#   - downstream hit interpretation is simply Unknown
# ============================================================================

unknown_code_mask = (
    dictionary_fg[
        "dictionary_resolution_status"
    ]
    .eq("UnknownCode")
    .fillna(False)
)

dictionary_fg.loc[
    unknown_code_mask,
    "hit_interpretation",
] = "Unknown"

dictionary_fg.loc[
    unknown_code_mask,
    "MOSAIC_FG",
] = pd.NA

dictionary_fg.loc[
    unknown_code_mask,
    "MOSAIC_FG_structural",
] = pd.NA


# ============================================================================
# 8. REASSERT UNKNOWNPLANT SEMANTICS
#
# UnknownPlant is different from UnknownCode:
#
# UnknownPlant:
#   We know it is vegetation, but cannot resolve it further.
#
# UnknownCode:
#   We cannot support even a biological interpretation.
# ============================================================================

unknown_plant_mask = (
    dictionary_fg[
        "dictionary_resolution_status"
    ]
    .eq("UnknownPlant")
    .fillna(False)
)

dictionary_fg.loc[
    unknown_plant_mask,
    "MOSAIC_FG_structural",
] = "UnknownPlant"

dictionary_fg.loc[
    unknown_plant_mask,
    "MOSAIC_FG",
] = "UnknownPlant"

dictionary_fg.loc[
    unknown_plant_mask,
    "hit_interpretation",
] = "UnknownPlant"


# ============================================================================
# 9. TERMINAL NON-NULL SAFETY
# ============================================================================

still_missing_status = (
    dictionary_fg[
        "dictionary_resolution_status"
    ]
    .isna()
)

dictionary_fg.loc[
    still_missing_status,
    "dictionary_resolution_status",
] = "UnknownCode"


still_missing_interpretation = (
    dictionary_fg[
        "hit_interpretation"
    ]
    .isna()
)

dictionary_fg.loc[
    still_missing_interpretation,
    "hit_interpretation",
] = "Unknown"


# ============================================================================
# 10. RECORD-WEIGHTED RESOLUTION SUMMARY
# ============================================================================

dictionary_fg["n_records_num"] = pd.to_numeric(
    dictionary_fg["n_records"],
    errors="coerce",
).fillna(0)

resolution_summary = (
    dictionary_fg
    .groupby(
        "dictionary_resolution_status",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

print("=" * 90)
print("UNIVERSAL DICTIONARY RESOLUTION")
print("=" * 90)

display(
    resolution_summary
)


# ============================================================================
# 11. HARD SEMANTIC ASSERTIONS
# ============================================================================

# Every dictionary row must retain its original observed code.
assert (
    dictionary_fg["observed_code"]
    .notna()
    .all()
), (
    "At least one dictionary row lacks observed_code."
)


# Every dictionary row must have a canonical code.
assert (
    dictionary_fg["canonical_code"]
    .notna()
    .all()
), (
    "At least one dictionary row lacks canonical_code."
)


# Every dictionary row must have a readable label.
assert (
    dictionary_fg["canonical_label"]
    .notna()
    .all()
), (
    "At least one dictionary row lacks canonical_label."
)


# Every dictionary row must have a resolution status.
assert (
    dictionary_fg[
        "dictionary_resolution_status"
    ]
    .notna()
    .all()
), (
    "At least one dictionary row lacks dictionary_resolution_status."
)


# Every dictionary row must have an interpretation.
assert (
    dictionary_fg[
        "hit_interpretation"
    ]
    .notna()
    .all()
), (
    "At least one dictionary row lacks hit_interpretation."
)


# Every plant must have some FG fallback.
assert (
    dictionary_fg.loc[
        is_plant,
        "MOSAIC_FG",
    ]
    .notna()
    .all()
), (
    "At least one Plant row lacks MOSAIC_FG."
)


# UnknownCode must not carry biological FG information.
assert (
    dictionary_fg.loc[
        unknown_code_mask,
        "hit_interpretation",
    ]
    .eq("Unknown")
    .all()
), (
    "At least one UnknownCode is not interpreted as Unknown."
)

assert (
    dictionary_fg.loc[
        unknown_code_mask,
        "MOSAIC_FG",
    ]
    .isna()
    .all()
), (
    "At least one UnknownCode still has a refined MOSAIC FG."
)

assert (
    dictionary_fg.loc[
        unknown_code_mask,
        "MOSAIC_FG_structural",
    ]
    .isna()
    .all()
), (
    "At least one UnknownCode still has a structural MOSAIC FG."
)


# UnknownPlant must remain explicitly interpretable as vegetation.
assert (
    dictionary_fg.loc[
        unknown_plant_mask,
        "MOSAIC_FG",
    ]
    .eq("UnknownPlant")
    .all()
), (
    "At least one UnknownPlant lacks UnknownPlant MOSAIC FG."
)

assert (
    dictionary_fg.loc[
        unknown_plant_mask,
        "hit_interpretation",
    ]
    .eq("UnknownPlant")
    .all()
), (
    "At least one UnknownPlant is not interpreted as UnknownPlant."
)


# ============================================================================
# 12. OPTIONAL ODDBALL DIAGNOSTIC
#
# GUTcfSAR-type codes must survive exactly as observed.
# canonical_code may differ only if we actually resolved them.
# ============================================================================

for oddball in [
    "GUTcfSAR",
]:

    odd = dictionary_fg.loc[
        dictionary_fg["observed_code"]
        .eq(oddball)
        .fillna(False)
    ]

    if len(odd) == 1:

        row = odd.iloc[0]

        assert (
            row["observed_code"] == oddball
        ), (
            f"Observed provenance was altered for {oddball}."
        )

        assert pd.notna(
            row["canonical_code"]
        )

        assert pd.notna(
            row["dictionary_resolution_status"]
        )

        assert pd.notna(
            row["hit_interpretation"]
        )

        print(
            f"\nOddball preserved: {oddball}"
        )

        display(
            odd[
                [
                    "observed_code",
                    "canonical_code",
                    "canonical_label",
                    "code_class",
                    "USDA_accepted_symbol",
                    "USDA_match_method",
                    "dictionary_resolution_status",
                    "MOSAIC_FG_structural",
                    "MOSAIC_FG",
                    "hit_interpretation",
                    "n_records_num",
                ]
            ]
        )


# ============================================================================
# 13. UNKNOWN TAIL SUMMARY
# ============================================================================

unknown_tail = dictionary_fg.loc[
    dictionary_fg[
        "dictionary_resolution_status"
    ].isin(
        [
            "UnknownPlant",
            "UnknownCode",
        ]
    )
].copy()

print("\n" + "=" * 90)
print("EXPLICIT UNKNOWN TAIL")
print("=" * 90)

display(
    unknown_tail[
        [
            "observed_code",
            "canonical_code",
            "canonical_label",
            "code_class",
            "dictionary_resolution_status",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "hit_interpretation",
            "n_records_num",
        ]
    ]
    .sort_values(
        "n_records_num",
        ascending=False,
    )
    .head(100)
)


print(
    "\nEvery observed dictionary code now has "
    "a complete and explicit interpretation."
)

UNIVERSAL DICTIONARY RESOLUTION


,dictionary_resolution_status,n_codes,n_records
1,NonPlant_classified,45,10270278
4,USDA_exact_taxon,7160,5162304
0,NoCanopy,1,442612
2,NonUSDA_plant_functionally_resolved,3002,198192
3,USDA_corrected_or_synonym_taxon,445,54731
6,UnknownPlant,142,21114
5,UnknownCode,165,870



Oddball preserved: GUTcfSAR


,observed_code,canonical_code,canonical_label,code_class,USDA_accepted_symbol,USDA_match_method,dictionary_resolution_status,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,n_records_num
1963,GUTcfSAR,GUSA2,Gutierrezia sarothrae (Pursh) Britton & Rusby,Plant,GUSA2,manual_alias,USDA_corrected_or_synonym_taxon,Woody,Woody,Woody,155



EXPLICIT UNKNOWN TAIL


,observed_code,canonical_code,canonical_label,code_class,dictionary_resolution_status,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,n_records_num
111,Plant base,Plant base,Plant base,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,17980
963,BOETRII,BOETRII,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,476
1865,PINSCO,PINSCO,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,441
1155,PHYGOR,PHYGOR,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,365
948,PHYFEN,PHYFEN,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,290
1790,SIDTEN,SIDTEN,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,246
1392,SPOCFCRY,SPOCFCRY,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,220
1224,Unk,Unk,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,195
1794,UNK,UNK,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,86
2786,BOERHAF,BOERHAF,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,75



Every observed dictionary code now has a complete and explicit interpretation.


In [112]:
# ============================================================================
# DIAGNOSE HOW "Plant base" STORES / LINKS PLANT IDENTITY
# ============================================================================

from pathlib import Path
import pandas as pd


LPI_GEO_FILE = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\LDC_LPI_georeferenced_2018_present.csv"
)

lpi = pd.read_csv(
    LPI_GEO_FILE,
    dtype="string",
    low_memory=False,
)

print(f"Total LPI rows: {len(lpi):,}")


# ============================================================================
# 1. FIND ALL PLANT-BASE OBSERVATIONS
# ============================================================================

plant_base = lpi.loc[
    lpi["code"]
    .astype("string")
    .str.strip()
    .eq("Plant base")
].copy()

print(
    "Plant base observations:",
    f"{len(plant_base):,}"
)

print("\nLayers containing Plant base:")

display(
    plant_base["layer"]
    .value_counts(dropna=False)
    .rename_axis("layer")
    .reset_index(name="n_records")
)


# ============================================================================
# 2. DEFINE PIN IDENTITY
#
# PrimaryKey = visit/event
# LineKey    = transect
# PointNbr   = pin
#
# This is the ecological sampling unit we need to inspect.
# ============================================================================

PIN_KEYS = [
    "PrimaryKey",
    "LineKey",
    "PointNbr",
]


# ============================================================================
# 3. PULL ALL ROWS FROM PINS THAT CONTAIN PLANT BASE
# ============================================================================

plant_base_pins = (
    plant_base[PIN_KEYS]
    .drop_duplicates()
)

pin_context = lpi.merge(
    plant_base_pins.assign(
        _plant_base_pin=True
    ),
    how="inner",
    on=PIN_KEYS,
    validate="many_to_one",
)


# ============================================================================
# 4. SHOW LAYERS / CODES ASSOCIATED WITH PLANT-BASE PINS
# ============================================================================

print("\nMost common code/layer combinations on Plant-base pins:")

display(
    pin_context
    .groupby(
        [
            "layer",
            "code",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="n")
    .sort_values(
        "n",
        ascending=False,
    )
    .head(100)
)


# ============================================================================
# 5. FOR EACH PLANT-BASE PIN, COUNT OTHER PLANT-LIKE HITS
#
# We are not yet deciding which one is the basal species.
# We are determining whether another code exists at the same pin.
# ============================================================================

context_no_base = pin_context.loc[
    ~(
        pin_context["code"]
        .astype("string")
        .str.strip()
        .eq("Plant base")
    )
].copy()


other_hits_per_pin = (
    context_no_base
    .groupby(
        PIN_KEYS,
        dropna=False,
    )
    .agg(
        n_other_rows=(
            "code",
            "size",
        ),
        n_other_codes=(
            "code",
            "nunique",
        ),
        other_codes=(
            "code",
            lambda x: " | ".join(
                sorted(
                    {
                        str(v).strip()
                        for v in x.dropna()
                        if str(v).strip()
                    }
                )
            ),
        ),
        other_layers=(
            "layer",
            lambda x: " | ".join(
                sorted(
                    {
                        str(v).strip()
                        for v in x.dropna()
                        if str(v).strip()
                    }
                )
            ),
        ),
    )
    .reset_index()
)


plant_base_context = plant_base_pins.merge(
    other_hits_per_pin,
    how="left",
    on=PIN_KEYS,
    validate="one_to_one",
)


# ============================================================================
# 6. SUMMARY
# ============================================================================

plant_base_context["has_other_hit"] = (
    plant_base_context[
        "n_other_rows"
    ]
    .fillna(0)
    .astype(float)
    .gt(0)
)

print("\nPlant-base pins with another associated row:")

display(
    plant_base_context[
        "has_other_hit"
    ]
    .value_counts(dropna=False)
    .rename_axis("has_other_hit")
    .reset_index(name="n_pins")
)


# ============================================================================
# 7. EXAMPLES
# ============================================================================

print("\nExample Plant-base pins and all associated observations:")

example_pins = (
    plant_base_pins
    .head(25)
)

examples = lpi.merge(
    example_pins.assign(
        _example=True
    ),
    how="inner",
    on=PIN_KEYS,
)

show_cols = [
    "PrimaryKey",
    "LineKey",
    "PointNbr",
    "RecKey",
    "layer",
    "code",
]

show_cols = [
    c for c in show_cols
    if c in examples.columns
]

display(
    examples[
        show_cols
    ]
    .sort_values(
        PIN_KEYS + ["layer"]
    )
)

Total LPI rows: 16,150,114
Plant base observations: 17,980

Layers containing Plant base:


,layer,n_records
0,SoilSurface,17980



Most common code/layer combinations on Plant-base pins:


,layer,code,n
49,SoilSurface,Plant base,17980
51,TopCanopy,Perennial grasses,4771
2,Lower1,Perennial grasses,3719
50,TopCanopy,Annual plants,2691
53,TopCanopy,Sub-shrubs and perennial forbs,1918
4,Lower1,Sub-shrubs and perennial forbs,1502
54,TopCanopy,Trees,1356
8,Lower2,HL,1173
52,TopCanopy,Shrubs,1145
9,Lower2,Perennial grasses,1106



Plant-base pins with another associated row:


,has_other_hit,n_pins
0,True,14775
1,False,3205



Example Plant-base pins and all associated observations:


,PrimaryKey,LineKey,PointNbr,RecKey,layer,code
12,26_mise_en_defens_planting_failed_2019-07-04_3...,EAST_20m,2,EAST_20m,SoilSurface,Plant base
54,26_mise_en_defens_planting_failed_2019-07-04_3...,EAST_20m,2,EAST_20m,TopCanopy,Perennial grasses
38,BalfL2_2023-01-14_4.84353_40.77134_s,EAST_25m,3,EAST_25m,Lower1,Perennial grasses
4,BalfL2_2023-01-14_4.84353_40.77134_s,EAST_25m,3,EAST_25m,SoilSurface,Plant base
33,BalfL2_2023-01-14_4.84353_40.77134_s,NORTH_25m,5,NORTH_25m,Lower1,Annual plants
14,BalfL2_2023-01-14_4.84353_40.77134_s,NORTH_25m,5,NORTH_25m,SoilSurface,Plant base
15,Boocame_0_2022-03-27_8.31069_47.80267_s,NORTH_20m,3,NORTH_20m,SoilSurface,Plant base
22,CSU18M07_2018-05-29_40.92888_-105.23833_l,NORTH_20m,3,NORTH_20m,SoilSurface,Plant base
47,CSU18M07_2018-05-29_40.92888_-105.23833_l,NORTH_20m,3,NORTH_20m,TopCanopy,Perennial grasses
51,H0904-1_2018-09-04_32.2751_-106.75053_l,EAST_20m,2,EAST_20m,Lower1,Shrubs


In [113]:
# ============================================================================
# IDENTIFY SURVEY SOURCES USING "Plant base"
# ============================================================================

import pandas as pd

# --------------------------------------------------------------------------
# 1. Isolate Plant-base observations
# --------------------------------------------------------------------------

plant_base = lpi.loc[
    lpi["code"]
    .astype("string")
    .str.strip()
    .eq("Plant base")
].copy()

print(
    "Plant base records:",
    f"{len(plant_base):,}"
)

assert (
    plant_base["layer"]
    .astype("string")
    .eq("SoilSurface")
    .all()
), (
    "Expected every Plant base record to occur in SoilSurface."
)


# ============================================================================
# 2. INSPECT AVAILABLE SOURCE / PROJECT FIELDS
# ============================================================================

source_cols = [
    c
    for c in [
        "source",
        "source.x",
        "source.y",
        "ProjectKey",
        "ProjectKey.x",
        "ProjectKey.y",
        "DBKey",
        "DBKey.x",
        "DBKey.y",
    ]
    if c in plant_base.columns
]

print("\nAvailable provenance fields:")
print(source_cols)


# ============================================================================
# 3. VALUE COUNTS FOR EACH PROVENANCE FIELD
# ============================================================================

for col in source_cols:

    print("\n" + "=" * 90)
    print(col)
    print("=" * 90)

    display(
        plant_base[col]
        .value_counts(
            dropna=False
        )
        .rename_axis(col)
        .reset_index(
            name="n_plant_base_records"
        )
        .head(100)
    )


# ============================================================================
# 4. UNIQUE VISITS / PROJECTS USING PLANT BASE
#
# PrimaryKey is the visit/event identifier, so this tells us whether Plant base
# is concentrated in a limited set of surveys rather than spread nationally.
# ============================================================================

visit_summary_cols = (
    source_cols
    + [
        c
        for c in [
            "PrimaryKey",
            "LineKey",
            "PointNbr",
        ]
        if c in plant_base.columns
    ]
)

visit_source_summary = (
    plant_base[
        visit_summary_cols
    ]
    .drop_duplicates()
)

print("\n" + "=" * 90)
print("PLANT-BASE VISITS BY SOURCE")
print("=" * 90)

if "PrimaryKey" in plant_base.columns:

    group_cols = [
        c
        for c in [
            "source.x",
            "source.y",
            "ProjectKey.x",
            "ProjectKey.y",
            "DBKey.x",
            "DBKey.y",
        ]
        if c in plant_base.columns
    ]

    if group_cols:

        source_visit_summary = (
            plant_base
            .groupby(
                group_cols,
                dropna=False,
            )
            .agg(
                n_records=(
                    "code",
                    "size",
                ),
                n_visits=(
                    "PrimaryKey",
                    "nunique",
                ),
                n_lines=(
                    "LineKey",
                    "nunique",
                ),
            )
            .reset_index()
            .sort_values(
                "n_records",
                ascending=False,
            )
        )

        display(
            source_visit_summary.head(100)
        )


# ============================================================================
# 5. VISIT-LEVEL LIST
#
# Useful for recognizing project naming patterns such as memze004,
# kweikonje_3, etc.
# ============================================================================

visit_cols = [
    c
    for c in [
        "PrimaryKey",
        "source.x",
        "source.y",
        "ProjectKey.x",
        "ProjectKey.y",
        "DBKey.x",
        "DBKey.y",
    ]
    if c in plant_base.columns
]

plant_base_visits = (
    plant_base[
        visit_cols
    ]
    .drop_duplicates()
)

print(
    "\nUnique visits containing Plant base:",
    f"{plant_base['PrimaryKey'].nunique():,}"
)

display(
    plant_base_visits
    .sort_values(
        "PrimaryKey"
    )
    .head(200)
)


# ============================================================================
# 6. PRIMARYKEY PREFIX DIAGNOSTIC
#
# If formal source fields are sparse or inconsistent, the PrimaryKey itself
# often exposes project/survey naming conventions.
# ============================================================================

plant_base["PrimaryKey_prefix"] = (
    plant_base["PrimaryKey"]
    .astype("string")
    .str.split("_")
    .str[0]
)

prefix_summary = (
    plant_base
    .groupby(
        "PrimaryKey_prefix",
        dropna=False,
    )
    .agg(
        n_records=(
            "code",
            "size",
        ),
        n_visits=(
            "PrimaryKey",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

print("\n" + "=" * 90)
print("PRIMARYKEY PREFIX SUMMARY")
print("=" * 90)

display(
    prefix_summary.head(100)
)


# ============================================================================
# 7. HOW CONCENTRATED IS PLANT BASE?
# ============================================================================

if len(prefix_summary) > 0:

    total_pb = prefix_summary["n_records"].sum()

    prefix_summary["pct_plant_base_records"] = (
        100
        * prefix_summary["n_records"]
        / total_pb
    )

    prefix_summary["cumulative_pct"] = (
        prefix_summary[
            "pct_plant_base_records"
        ]
        .cumsum()
    )

    print("\nConcentration of Plant-base usage:")

    display(
        prefix_summary.head(50)
    )

Plant base records: 17,980

Available provenance fields:
['source.x', 'source.y', 'ProjectKey.x', 'ProjectKey.y', 'DBKey.x', 'DBKey.y']

source.x


,source.x,n_plant_base_records
0,LandPKS,17980



source.y


,source.y,n_plant_base_records
0,LandPKS,17980



ProjectKey.x


...

,PrimaryKey,source.x,source.y,ProjectKey.x,ProjectKey.y,DBKey.x,DBKey.y
12828740,003Klfv2_2018-05-30_-3.41437_39.49765_s,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12905648,008MalkagufuV_2018-07-04_2.61351_39.57818_b,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12777089,008WichirV_2018-07-03_1.71342_40.00964_b,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12884958,017_mkn1v_2018-06-27_-2.09593_37.53674_p,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12997467,017mkn2v_2018-06-22_-1.69332_37.59623_p,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12862962,017mknv_2018-06-16_-2.09589_37.53714_p,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
13055199,02kwl4v_2018-06-15_-4.3316_39.14068_j,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12862599,04-Ntungamo-WRZ-334_2022-05-04_-0.85425_30.167...,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12732055,05-LCVZ-KYANKWANZI-001_2022-05-05_1.10081_31.5...,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09
12961758,05-LVCZ-KYANKWANZI-001_2022-05-05_1.10086_31.5...,LandPKS,LandPKS,LandPKS,LandPKS,LandPKS_2025-05-09,LandPKS_2025-05-09



PRIMARYKEY PREFIX SUMMARY


,PrimaryKey_prefix,n_records,n_visits
724,menze,762,10
370,LALAIMURUNYAI,292,7
758,oltepesi,195,2
752,near,188,3
521,PSE,166,13
420,MALISHO,155,3
338,ILOSUKUTI,145,5
275,EBOLEYI,128,7
280,ESILALEI,123,6
478,NGOULO,123,3



Concentration of Plant-base usage:


,PrimaryKey_prefix,n_records,n_visits,pct_plant_base_records,cumulative_pct
724,menze,762,10,4.238042,4.238042
370,LALAIMURUNYAI,292,7,1.624027,5.862069
758,oltepesi,195,2,1.084538,6.946607
752,near,188,3,1.045606,7.992214
521,PSE,166,13,0.923248,8.915462
420,MALISHO,155,3,0.862069,9.777531
338,ILOSUKUTI,145,5,0.806452,10.583982
275,EBOLEYI,128,7,0.711902,11.295884
280,ESILALEI,123,6,0.684093,11.979978
478,NGOULO,123,3,0.684093,12.664071


In [114]:
# ============================================================================
# 20b. EXPLICIT PROTOCOL OVERRIDE: LANDPKS "Plant base"
# ============================================================================
#
# "Plant base" is a LandPKS SoilSurface protocol observation indicating
# basal vegetation contact.
#
# It is NOT:
#   - a taxon
#   - an UnknownPlant
#   - a MOSAIC functional group
#
# It should therefore be carried as its own protocol class.
# ============================================================================

plant_base_mask = (
    dictionary_fg["observed_code"]
    .astype("string")
    .str.strip()
    .eq("Plant base")
    .fillna(False)
)

assert plant_base_mask.sum() == 1, (
    f"Expected exactly one Plant base dictionary row; "
    f"found {plant_base_mask.sum()}."
)


# ----------------------------------------------------------------------------
# Core protocol classification
# ----------------------------------------------------------------------------

dictionary_fg.loc[
    plant_base_mask,
    "code_class",
] = "PlantBase"

dictionary_fg.loc[
    plant_base_mask,
    "canonical_code",
] = "Plant base"

dictionary_fg.loc[
    plant_base_mask,
    "canonical_label",
] = "Plant basal contact"

dictionary_fg.loc[
    plant_base_mask,
    "dictionary_resolution_status",
] = "ProtocolPlantBase"

dictionary_fg.loc[
    plant_base_mask,
    "hit_interpretation",
] = "PlantBase"


# ----------------------------------------------------------------------------
# Explicitly remove unsupported biological interpretation
# ----------------------------------------------------------------------------

dictionary_fg.loc[
    plant_base_mask,
    "MOSAIC_FG_structural",
] = pd.NA

dictionary_fg.loc[
    plant_base_mask,
    "MOSAIC_FG",
] = pd.NA


# ----------------------------------------------------------------------------
# Optional provenance fields
#
# Preserve the original resolution history where possible, but make clear
# that this final interpretation is an explicit protocol override.
# ----------------------------------------------------------------------------

if "resolution_source" in dictionary_fg.columns:
    dictionary_fg.loc[
        plant_base_mask,
        "resolution_source",
    ] = "LandPKS_protocol_override"

if "FG_resolution_source" in dictionary_fg.columns:
    dictionary_fg.loc[
        plant_base_mask,
        "FG_resolution_source",
    ] = "not_applicable_protocol_plant_base"


# ----------------------------------------------------------------------------
# Hard checks
# ----------------------------------------------------------------------------

pb = dictionary_fg.loc[
    plant_base_mask
].iloc[0]

assert pb["observed_code"] == "Plant base"

assert pb["code_class"] == "PlantBase"

assert pb["canonical_code"] == "Plant base"

assert pb["canonical_label"] == "Plant basal contact"

assert (
    pb["dictionary_resolution_status"]
    == "ProtocolPlantBase"
)

assert pb["hit_interpretation"] == "PlantBase"

assert pd.isna(
    pb["MOSAIC_FG_structural"]
)

assert pd.isna(
    pb["MOSAIC_FG"]
)


print(
    "LandPKS Plant base override applied:"
)

display(
    dictionary_fg.loc[
        plant_base_mask,
        [
            "observed_code",
            "code_class",
            "canonical_code",
            "canonical_label",
            "dictionary_resolution_status",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "hit_interpretation",
            "resolution_source",
            "FG_resolution_source",
            "n_records",
        ],
    ]
)

LandPKS Plant base override applied:


,observed_code,code_class,canonical_code,canonical_label,dictionary_resolution_status,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,resolution_source,FG_resolution_source,n_records
111,Plant base,PlantBase,Plant base,Plant basal contact,ProtocolPlantBase,<NA>,<NA>,PlantBase,LandPKS_protocol_override,not_applicable_protocol_plant_base,17980


In [115]:
# ============================================================================
# 20c. EXPLICIT TAXONOMIC + ECOLOGICAL OVERRIDE: BASSIA CODES
# ============================================================================
#
# Raw LPI source codes:
#   BAPRV
#   BAPRG
#
# Canonical USDA identity:
#   BAPR5
#
# MOSAIC ecological interpretation:
#   ExoticForb
#
# observed_code remains unchanged for provenance.
# canonical_code is redirected to BAPR5 for downstream LPI composition.
# ============================================================================

BASSIA_CODES = [
    "BAPRV",
    "BAPRG",
]

bassia_mask = (
    dictionary_fg["observed_code"]
    .astype("string")
    .isin(BASSIA_CODES)
)

assert bassia_mask.sum() == 2, (
    f"Expected exactly two Bassia rows "
    f"{BASSIA_CODES}; found {bassia_mask.sum()}."
)


# ============================================================================
# 1. TAXONOMIC REDIRECTION
# ============================================================================

dictionary_fg.loc[
    bassia_mask,
    "USDA_accepted_symbol",
] = "BAPR5"

dictionary_fg.loc[
    bassia_mask,
    "canonical_code",
] = "BAPR5"

dictionary_fg.loc[
    bassia_mask,
    "USDA_match_method",
] = "manual_synonym_resolution"

dictionary_fg.loc[
    bassia_mask,
    "resolution_source",
] = "manual_USDA_synonym_resolution"

dictionary_fg.loc[
    bassia_mask,
    "code_class",
] = "Plant"

dictionary_fg.loc[
    bassia_mask,
    "dictionary_resolution_status",
] = "USDA_corrected_or_synonym_taxon"


# ============================================================================
# 2. PULL ACCEPTED BAPR5 TAXONOMY
#
# Use the same accepted USDA lookup used elsewhere in the notebook.
# ============================================================================

bapr5 = usda_accepted_lookup.loc[
    usda_accepted_lookup[
        "USDA_accepted_symbol"
    ].eq("BAPR5")
].copy()

assert len(bapr5) == 1, (
    f"Expected exactly one accepted USDA row for BAPR5; "
    f"found {len(bapr5)}."
)

bapr5 = bapr5.iloc[0]


for col in [
    "USDA_scientific_name",
    "USDA_common_name",
    "USDA_family",
]:

    if (
        col in dictionary_fg.columns
        and col in bapr5.index
    ):

        dictionary_fg.loc[
            bassia_mask,
            col,
        ] = bapr5[col]


# Canonical label should describe the accepted BAPR5 taxon.
if pd.notna(
    bapr5.get("USDA_scientific_name")
):

    dictionary_fg.loc[
        bassia_mask,
        "canonical_label",
    ] = bapr5[
        "USDA_scientific_name"
    ]


# ============================================================================
# 3. MOSAIC ECOLOGICAL OVERRIDE
#
# USDA may represent Bassia with woody/subshrub growth-habit information,
# but these LPI observations are intentionally treated as exotic forbs.
# ============================================================================

dictionary_fg.loc[
    bassia_mask,
    "MOSAIC_FG_structural",
] = "Forb"

dictionary_fg.loc[
    bassia_mask,
    "MOSAIC_FG",
] = "ExoticForb"

dictionary_fg.loc[
    bassia_mask,
    "hit_interpretation",
] = "ExoticForb"

dictionary_fg.loc[
    bassia_mask,
    "FG_resolution_source",
] = "manual_ecological_functional_override"


# ============================================================================
# 4. HARD CHECKS
# ============================================================================

for code in BASSIA_CODES:

    row = dictionary_fg.loc[
        dictionary_fg[
            "observed_code"
        ]
        .astype("string")
        .eq(code)
        .fillna(False)
    ]

    assert len(row) == 1, (
        f"Expected one row for {code}; "
        f"found {len(row)}."
    )

    row = row.iloc[0]

    # Raw provenance remains intact.
    assert row["observed_code"] == code

    # Both source codes canonicalize to BAPR5.
    assert (
        row["USDA_accepted_symbol"]
        == "BAPR5"
    )

    assert (
        row["canonical_code"]
        == "BAPR5"
    )

    assert (
        row[
            "dictionary_resolution_status"
        ]
        == "USDA_corrected_or_synonym_taxon"
    )

    # Ecological interpretation is explicit.
    assert (
        row["MOSAIC_FG_structural"]
        == "Forb"
    )

    assert (
        row["MOSAIC_FG"]
        == "ExoticForb"
    )

    assert (
        row["hit_interpretation"]
        == "ExoticForb"
    )

    assert (
        row["FG_resolution_source"]
        == "manual_ecological_functional_override"
    )


print(
    "Bassia codes redirected:"
)

display(
    dictionary_fg.loc[
        bassia_mask,
        [
            "observed_code",
            "canonical_code",
            "USDA_accepted_symbol",
            "USDA_scientific_name",
            "USDA_common_name",
            "dictionary_resolution_status",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "hit_interpretation",
            "FG_resolution_source",
            "n_records",
        ],
    ]
)

Bassia codes redirected:


,observed_code,canonical_code,USDA_accepted_symbol,USDA_scientific_name,USDA_common_name,dictionary_resolution_status,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,FG_resolution_source,n_records
4193,BAPRV,BAPR5,BAPR5,Bassia prostrata (L.) A.J. Scott,forage kochia,USDA_corrected_or_synonym_taxon,Forb,ExoticForb,ExoticForb,manual_ecological_functional_override,14
5664,BAPRG,BAPR5,BAPR5,Bassia prostrata (L.) A.J. Scott,forage kochia,USDA_corrected_or_synonym_taxon,Forb,ExoticForb,ExoticForb,manual_ecological_functional_override,4


In [116]:
# ============================================================================
# 20d. NORMALIZE PURE USDA LICHENOUS TAXA
# ============================================================================
#
# Pure USDA growth habit "Lichenous" should resolve to Lichen whenever no
# stronger manual/protocol ecological override has been applied.
#
# Preserve explicit manual/project overrides.
# ============================================================================

lichenous_mask = (
    dictionary_fg["USDA_growth_habit"]
    .astype("string")
    .eq("Lichenous")
    .fillna(False)
)

print(
    "Pure USDA Lichenous rows:",
    f"{lichenous_mask.sum():,}"
)


# ============================================================================
# 1. IDENTIFY STRONGER OVERRIDES
# ============================================================================

fg_source = (
    dictionary_fg["FG_resolution_source"]
    .astype("string")
)

strong_override = (
    fg_source.str.contains(
        r"manual|protocol|project",
        case=False,
        na=False,
        regex=True,
    )
)


# ============================================================================
# 2. APPLY USDA LICHEN FALLBACK
#
# Only rows without stronger explicit ecological overrides are normalized.
# ============================================================================

lichen_fallback_mask = (
    lichenous_mask
    & ~strong_override
)

dictionary_fg.loc[
    lichen_fallback_mask,
    "MOSAIC_FG_structural",
] = "Lichen"

dictionary_fg.loc[
    lichen_fallback_mask,
    "MOSAIC_FG",
] = "Lichen"

dictionary_fg.loc[
    lichen_fallback_mask,
    "hit_interpretation",
] = "Lichen"


# Record the provenance of the derived classification where appropriate.
dictionary_fg.loc[
    lichen_fallback_mask,
    "FG_resolution_source",
] = "USDA_growth_habit"


# ============================================================================
# 3. INSPECT ANY LICHENOUS ROWS NOT NORMALIZED
# ============================================================================

lichen_override_rows = dictionary_fg.loc[
    lichenous_mask
    & strong_override,
    [
        "observed_code",
        "USDA_accepted_symbol",
        "USDA_growth_habit",
        "protocol_plant_group",
        "MOSAIC_FG_structural",
        "MOSAIC_FG",
        "FG_resolution_source",
        "n_records",
    ],
].copy()

print(
    "Lichenous rows retaining stronger overrides:",
    f"{len(lichen_override_rows):,}"
)

if len(lichen_override_rows) > 0:
    display(
        lichen_override_rows.sort_values(
            "n_records",
            ascending=False,
        )
    )


# ============================================================================
# 4. HARD CHECK FOR USDA-DERIVED LICHEN ROWS
# ============================================================================

assert (
    dictionary_fg.loc[
        lichen_fallback_mask,
        "MOSAIC_FG_structural",
    ]
    .eq("Lichen")
    .all()
), (
    "At least one USDA-derived pure Lichenous row "
    "did not resolve structurally to Lichen."
)

assert (
    dictionary_fg.loc[
        lichen_fallback_mask,
        "MOSAIC_FG",
    ]
    .eq("Lichen")
    .all()
), (
    "At least one USDA-derived pure Lichenous row "
    "did not resolve to Lichen."
)


print(
    "Pure USDA Lichenous fallback normalized to Lichen."
)

Pure USDA Lichenous rows: 70
Lichenous rows retaining stronger overrides: 0
Pure USDA Lichenous fallback normalized to Lichen.


In [117]:
# ============================================================================
# BIENNIAL FORB SANITY CHECK
#
# Pure USDA Biennial + Forb/herb should resolve to BiennialForb unless a
# higher-priority manual / protocol / project ecological classification exists.
# ============================================================================

biennial_forb_rows = final.loc[
    final["USDA_duration"]
    .astype("string")
    .eq("Biennial")
    .fillna(False)
    &
    final["USDA_growth_habit"]
    .astype("string")
    .isin(
        [
            "Forb/herb",
            "Forb/herb|Vine",
        ]
    )
].copy()


if len(biennial_forb_rows) > 0:

    biennial_has_override = (
        biennial_forb_rows[
            "FG_resolution_source"
        ]
        .astype("string")
        .str.contains(
            r"manual|protocol|project",
            case=False,
            na=False,
            regex=True,
        )
    )

    usda_biennial_forbs = (
        biennial_forb_rows.loc[
            ~biennial_has_override
        ]
    )

    # Diagnostic first so failure is interpretable.
    bad_biennials = (
        usda_biennial_forbs.loc[
            ~(
                usda_biennial_forbs[
                    "MOSAIC_FG_structural"
                ]
                .eq("BiennialForb")
                .fillna(False)
            )
        ]
    )

    if len(bad_biennials) > 0:

        print(
            f"Normalizing {len(bad_biennials):,} "
            "USDA-derived pure Biennial forb rows."
        )

        display(
            bad_biennials[
                [
                    "observed_code",
                    "USDA_accepted_symbol",
                    "USDA_duration",
                    "USDA_growth_habit",
                    "protocol_plant_group",
                    "MOSAIC_FG_structural",
                    "MOSAIC_FG",
                    "FG_resolution_source",
                    "n_records",
                ]
            ]
        )

        bad_idx = bad_biennials.index

        final.loc[
            bad_idx,
            "MOSAIC_FG_structural",
        ] = "BiennialForb"

        # Preserve nativity refinement where available.
        native = (
            final.loc[
                bad_idx,
                "USDA_native_status_L48",
            ]
            .astype("string")
        )

        final.loc[
            bad_idx,
            "MOSAIC_FG",
        ] = "BiennialForb"

        final.loc[
            bad_idx[
                native.eq("I").fillna(False).to_numpy()
            ],
            "MOSAIC_FG",
        ] = "ExoticBiennialForb"

        final.loc[
            bad_idx[
                native.eq("N").fillna(False).to_numpy()
            ],
            "MOSAIC_FG",
        ] = "NativeBiennialForb"

        final.loc[
            bad_idx,
            "hit_interpretation",
        ] = final.loc[
            bad_idx,
            "MOSAIC_FG",
        ]

        final.loc[
            bad_idx,
            "FG_resolution_source",
        ] = "USDA_duration_growth_habit"


    # Final check only on rows where USDA is actually the governing source.
    assert (
        final.loc[
            usda_biennial_forbs.index,
            "MOSAIC_FG_structural",
        ]
        .eq("BiennialForb")
        .fillna(False)
        .all()
    ), (
        "USDA-derived pure Biennial forb normalization failed."
    )

Normalizing 51 USDA-derived pure Biennial forb rows.


,observed_code,USDA_accepted_symbol,USDA_duration,USDA_growth_habit,protocol_plant_group,MOSAIC_FG_structural,MOSAIC_FG,FG_resolution_source,n_records
566,DACA6,DACA6,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,477
582,ERDI4,ERDI4,Biennial,Forb/herb,<NA>,Forb,NativeForb,USDA_two_level_traits,374
630,CIVU,CIVU,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,239
714,VETH,VETH,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,189
981,CYOF,CYOF,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,198
1049,ERFL,ERFL,Biennial,Forb/herb,<NA>,Forb,NativeForb,USDA_two_level_traits,109
1450,DEIN13,DEINI2,Biennial,Forb/herb,<NA>,Forb,NativeForb,USDA_two_level_traits,108
1488,ONAC,ONAC,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,122
1527,TRPO,TRPO,Biennial,Forb/herb,<NA>,Forb,ExoticForb,USDA_two_level_traits,83
1680,ARCA,ARCA,Biennial,Forb/herb,<NA>,Forb,NativeForb,USDA_two_level_traits,44


In [118]:
# ============================================================================
# 20e. TARGETED ECOLOGICAL OVERRIDE: DACAS2
# ============================================================================
#
# USDA traits for DACAS2 are:
#   Duration     = Biennial
#   Growth habit = Forb/herb
#
# Therefore its structural class is explicitly BiennialForb.
# Nativity is unavailable, so the refined MOSAIC class remains BiennialForb.
# ============================================================================

dacas2_mask = (
    dictionary_fg["observed_code"]
    .astype("string")
    .eq("DACAS2")
    .fillna(False)
)

assert dacas2_mask.sum() == 1, (
    f"Expected exactly one DACAS2 row; "
    f"found {dacas2_mask.sum()}."
)

# Confirm that the source USDA traits support this override.
dacas2 = dictionary_fg.loc[
    dacas2_mask
].iloc[0]

assert dacas2["USDA_duration"] == "Biennial", (
    f"DACAS2 expected USDA_duration='Biennial'; "
    f"found {dacas2['USDA_duration']!r}."
)

assert dacas2["USDA_growth_habit"] == "Forb/herb", (
    f"DACAS2 expected USDA_growth_habit='Forb/herb'; "
    f"found {dacas2['USDA_growth_habit']!r}."
)

# Apply explicit ecological interpretation.
dictionary_fg.loc[
    dacas2_mask,
    "MOSAIC_FG_structural",
] = "BiennialForb"

dictionary_fg.loc[
    dacas2_mask,
    "MOSAIC_FG",
] = "BiennialForb"

dictionary_fg.loc[
    dacas2_mask,
    "resolved_plant_group",
] = "BiennialForb"

dictionary_fg.loc[
    dacas2_mask,
    "hit_interpretation",
] = "BiennialForb"

dictionary_fg.loc[
    dacas2_mask,
    "FG_resolution_source",
] = "manual_ecological_functional_override"


# ============================================================================
# HARD CHECK
# ============================================================================

dacas2 = dictionary_fg.loc[
    dacas2_mask
].iloc[0]

assert dacas2["MOSAIC_FG_structural"] == "BiennialForb"
assert dacas2["MOSAIC_FG"] == "BiennialForb"
assert dacas2["hit_interpretation"] == "BiennialForb"

print("DACAS2 explicitly resolved to BiennialForb.")

display(
    dictionary_fg.loc[
        dacas2_mask,
        [
            "observed_code",
            "USDA_accepted_symbol",
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "hit_interpretation",
            "FG_resolution_source",
            "n_records",
        ],
    ]
)

DACAS2 explicitly resolved to BiennialForb.


,observed_code,USDA_accepted_symbol,USDA_duration,USDA_growth_habit,USDA_native_status_L48,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,FG_resolution_source,n_records
5812,DACAS2,DACAS2,Biennial,Forb/herb,<NA>,BiennialForb,BiennialForb,BiennialForb,manual_ecological_functional_override,2


In [119]:
print(structural_from_usda("Forb/herb", "Biennial"))

Forb


## 21. Hard QA gate

This is the only gate allowed to precede the final write. It checks code-universe accounting, record accounting, sentinel separation, the POAR2R2 correction, canonical nativity, and several ecological sentinel taxa including BRTE.


In [122]:
# ============================================================================
# 21. HARD FINAL QA/QC
# ============================================================================

raw = pd.read_csv(
    LPI_CODE_FILE,
    dtype={"code": "string"},
    low_memory=False,
).copy()

raw["observed_code_raw"] = raw["code"]

raw["source_code_was_blank"] = (
    raw["observed_code_raw"].isna()
    | raw["observed_code_raw"].str.strip().eq("")
)

raw["observed_code"] = (
    raw["observed_code_raw"]
    .astype("string")
    .str.strip()
)

raw.loc[
    raw["source_code_was_blank"],
    "observed_code",
] = "__NO_CANOPY__"


final = dictionary_fg.copy()


# ============================================================================
# 1. STRUCTURE / CODE-UNIVERSE ACCOUNTING
# ============================================================================

assert len(final) == len(raw), (
    f"Row-count mismatch: "
    f"raw={len(raw):,}, "
    f"final={len(final):,}"
)

assert (
    final["observed_code"]
    .nunique(dropna=False)
    == len(final)
), (
    "Final dictionary contains duplicate observed_code values."
)


raw_set = set(
    raw["observed_code"]
    .dropna()
    .astype(str)
)

final_set = set(
    final["observed_code"]
    .dropna()
    .astype(str)
)

assert raw_set == final_set, (
    "Code-universe mismatch. "
    f"Missing={len(raw_set - final_set):,}, "
    f"extra={len(final_set - raw_set):,}"
)


# --------------------------------------------------------------------------
# Record-weight accounting
# --------------------------------------------------------------------------

raw_records = (
    pd.to_numeric(
        raw["n_records"],
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

final_records = (
    pd.to_numeric(
        final["n_records"],
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

assert raw_records == final_records, (
    "Record accounting differs: "
    f"raw={raw_records:,.0f}, "
    f"final={final_records:,.0f}"
)


# ============================================================================
# 2. REQUIRED UNIVERSAL INTERPRETATION FIELDS
# ============================================================================

required_universal_cols = [
    "canonical_code",
    "canonical_label",
    "dictionary_resolution_status",
    "hit_interpretation",
]

for col in required_universal_cols:

    assert col in final.columns, (
        f"Missing required final column: {col}"
    )

    assert final[col].notna().all(), (
        f"{col} contains null values."
    )


# Every observed source code must have exactly one interpretation.
assert (
    final["observed_code"]
    .nunique(dropna=False)
    ==
    final["hit_interpretation"]
    .notna()
    .sum()
), (
    "Not every observed dictionary code has "
    "exactly one hit interpretation."
)


# ============================================================================
# 3. CORE RESOLUTION-STATUS SEMANTICS
# ============================================================================

allowed_resolution_status = {
    "USDA_exact_taxon",
    "USDA_corrected_or_synonym_taxon",
    "NonUSDA_plant_functionally_resolved",
    "UnknownPlant",
    "NoCanopy",
    "NonPlant_classified",
    "UnknownCode",
    "ProtocolPlantBase",
}

bad_status = final.loc[
    ~final[
        "dictionary_resolution_status"
    ].isin(
        allowed_resolution_status
    )
]

assert len(bad_status) == 0, (
    "Unexpected dictionary_resolution_status values remain:\n"
    f"{sorted(bad_status['dictionary_resolution_status'].dropna().unique())}"
)


# ============================================================================
# 4. PLANT ROWS
#
# True Plant rows must always have a functional-group interpretation,
# even if the best available value is UnknownPlant.
#
# PlantBase is intentionally NOT a Plant row.
# ============================================================================

plant_mask = (
    final["code_class"]
    .eq("Plant")
    .fillna(False)
)

assert (
    final.loc[
        plant_mask,
        "MOSAIC_FG",
    ]
    .notna()
    .all()
), (
    "At least one Plant row lacks MOSAIC_FG."
)

assert (
    final.loc[
        plant_mask,
        "MOSAIC_FG_structural",
    ]
    .notna()
    .all()
), (
    "At least one Plant row lacks MOSAIC_FG_structural."
)


# ============================================================================
# 5. UNKNOWNPLANT SEMANTICS
#
# UnknownPlant means:
#   - known to represent vegetation
#   - insufficient information for more specific taxonomic / FG assignment
# ============================================================================

unknown_plant_mask = (
    final[
        "dictionary_resolution_status"
    ]
    .eq("UnknownPlant")
    .fillna(False)
)

assert (
    final.loc[
        unknown_plant_mask,
        "code_class",
    ]
    .eq("Plant")
    .all()
), (
    "At least one UnknownPlant row is not classified as Plant."
)

assert (
    final.loc[
        unknown_plant_mask,
        "MOSAIC_FG_structural",
    ]
    .eq("UnknownPlant")
    .all()
), (
    "At least one UnknownPlant row lacks structural UnknownPlant."
)

assert (
    final.loc[
        unknown_plant_mask,
        "MOSAIC_FG",
    ]
    .eq("UnknownPlant")
    .all()
), (
    "At least one UnknownPlant row lacks refined UnknownPlant."
)

assert (
    final.loc[
        unknown_plant_mask,
        "hit_interpretation",
    ]
    .eq("UnknownPlant")
    .all()
), (
    "At least one UnknownPlant row is not interpreted as UnknownPlant."
)


# ============================================================================
# 6. UNKNOWNCODE SEMANTICS
#
# UnknownCode means:
#   - source code is preserved
#   - no defensible biological / protocol interpretation exists
#   - no functional group is asserted
# ============================================================================

unknown_code_mask = (
    final[
        "dictionary_resolution_status"
    ]
    .eq("UnknownCode")
    .fillna(False)
)

assert (
    final.loc[
        unknown_code_mask,
        "hit_interpretation",
    ]
    .eq("Unknown")
    .all()
), (
    "At least one UnknownCode is not interpreted as Unknown."
)

assert (
    final.loc[
        unknown_code_mask,
        "MOSAIC_FG_structural",
    ]
    .isna()
    .all()
), (
    "At least one UnknownCode retains a structural MOSAIC FG."
)

assert (
    final.loc[
        unknown_code_mask,
        "MOSAIC_FG",
    ]
    .isna()
    .all()
), (
    "At least one UnknownCode retains a refined MOSAIC FG."
)


# ============================================================================
# 7. PLANT BASE — LANDPKS PROTOCOL CLASS
#
# "Plant base" is a LandPKS SoilSurface protocol observation indicating
# basal vegetation contact. It does NOT encode species or FG identity.
# ============================================================================

plant_base = final.loc[
    final["observed_code"]
    .eq("Plant base")
    .fillna(False)
]

assert len(plant_base) == 1, (
    f"Expected exactly one Plant base dictionary row; "
    f"found {len(plant_base)}."
)

plant_base = plant_base.iloc[0]

assert plant_base["code_class"] == "PlantBase", (
    "Plant base must have code_class='PlantBase'."
)

assert (
    plant_base["dictionary_resolution_status"]
    == "ProtocolPlantBase"
), (
    "Plant base must have "
    "dictionary_resolution_status='ProtocolPlantBase'."
)

assert plant_base["canonical_code"] == "Plant base"

assert plant_base["canonical_label"] == "Plant basal contact"

assert plant_base["hit_interpretation"] == "PlantBase"

assert pd.isna(
    plant_base["MOSAIC_FG_structural"]
), (
    "Plant base must not have a structural functional group."
)

assert pd.isna(
    plant_base["MOSAIC_FG"]
), (
    "Plant base must not have a refined functional group."
)

assert int(
    pd.to_numeric(
        plant_base["n_records"],
        errors="raise",
    )
) == 17980, (
    "Unexpected Plant base record count."
)


# ============================================================================
# 8. __NO_CANOPY__ AND LITERAL USDA SYMBOL NONE
# ============================================================================

no_canopy = final.loc[
    final["observed_code"]
    .eq("__NO_CANOPY__")
    .fillna(False)
]

literal_none = final.loc[
    final["observed_code"]
    .eq("NONE")
    .fillna(False)
]

assert len(no_canopy) == 1, (
    "Expected exactly one __NO_CANOPY__ row."
)

assert len(literal_none) == 1, (
    "Expected exactly one literal NONE row."
)

no_canopy = no_canopy.iloc[0]
literal_none = literal_none.iloc[0]


assert no_canopy["code_class"] == "NoCanopy"

assert (
    no_canopy[
        "dictionary_resolution_status"
    ]
    == "NoCanopy"
)

assert no_canopy["hit_interpretation"] == "NO_CANOPY"

assert pd.isna(
    no_canopy["USDA_accepted_symbol"]
)


# Literal NONE is a real USDA plant symbol.
assert literal_none["code_class"] == "Plant"

assert (
    literal_none["USDA_accepted_symbol"]
    == "NONE"
)


# ============================================================================
# 9. KNOWN TAXONOMIC CORRECTION: POAR2R2 -> PONIN
# ============================================================================

poar = final.loc[
    final["observed_code"]
    .eq("POAR2R2")
    .fillna(False)
]

assert len(poar) == 1, (
    "Expected exactly one POAR2R2 row."
)

poar = poar.iloc[0]

assert poar["observed_code"] == "POAR2R2"

assert (
    poar["USDA_accepted_symbol"]
    == "PONIN"
)

assert (
    poar["canonical_code"]
    == "PONIN"
)

assert (
    poar["dictionary_resolution_status"]
    == "USDA_corrected_or_synonym_taxon"
)


assert not (
    final["USDA_accepted_symbol"]
    .eq("POAR22")
    .fillna(False)
    .any()
), (
    "Obsolete accepted symbol POAR22 remains."
)


# ============================================================================
# 10. CANONICAL NATIVITY ENCODING
# ============================================================================

bad_nat = final.loc[
    final["USDA_native_status_L48"].notna()
    &
    ~final[
        "USDA_native_status_L48"
    ].isin(
        [
            "N",
            "I",
            "I|N",
            "N|I",
        ]
    )
]

assert len(bad_nat) == 0, (
    "Noncanonical values remain in USDA_native_status_L48."
)


# ============================================================================
# 11. ODDBALL-CODE PRESERVATION
#
# observed_code is immutable provenance.
# canonical_code may legitimately differ if a code was resolved.
# ============================================================================

for oddball in [
    "GUTcfSAR",
]:

    x = final.loc[
        final["observed_code"]
        .eq(oddball)
        .fillna(False)
    ]

    if len(x) == 1:

        row = x.iloc[0]

        assert (
            row["observed_code"]
            == oddball
        ), (
            f"Observed provenance altered for {oddball}."
        )

        assert pd.notna(
            row["canonical_code"]
        ), (
            f"{oddball} lacks canonical_code."
        )

        assert pd.notna(
            row["dictionary_resolution_status"]
        ), (
            f"{oddball} lacks dictionary_resolution_status."
        )

        assert pd.notna(
            row["hit_interpretation"]
        ), (
            f"{oddball} lacks hit_interpretation."
        )

        print(
            f"Oddball code retained: "
            f"{oddball} -> "
            f"{row['dictionary_resolution_status']} / "
            f"{row['hit_interpretation']}"
        )


# ============================================================================
# 12. ECOLOGICAL SENTINEL TAXA
# ============================================================================

def one(code):

    x = final.loc[
        final["observed_code"]
        .eq(code)
        .fillna(False)
    ]

    assert len(x) == 1, (
        f"Expected one row for {code}; "
        f"found {len(x)}."
    )

    return x.iloc[0]


# --------------------------------------------------------------------------
# Annual exotic grass
# --------------------------------------------------------------------------

brte = one("BRTE")

assert (
    brte["MOSAIC_FG_structural"]
    == "AnnualGrass"
)

assert (
    brte["USDA_native_status_L48"]
    == "I"
)

assert (
    brte["MOSAIC_FG"]
    == "EAG"
)


# --------------------------------------------------------------------------
# Native perennial graminoid
# --------------------------------------------------------------------------

pose = one("POSE")

assert (
    pose["MOSAIC_FG_structural"]
    == "PerennialGrass"
)

assert (
    pose["USDA_native_status_L48"]
    == "N"
)

assert (
    pose["MOSAIC_FG"]
    == "NPG"
)


# --------------------------------------------------------------------------
# Exotic annual forb
# --------------------------------------------------------------------------

alde = one("ALDE")

assert (
    alde["MOSAIC_FG_structural"]
    == "AnnualForb"
)

assert (
    alde["MOSAIC_FG"]
    == "ExoticAnnualForb"
)


# --------------------------------------------------------------------------
# Native perennial forb
# --------------------------------------------------------------------------

phho = one("PHHO")

assert (
    phho["MOSAIC_FG_structural"]
    == "PerennialForb"
)

assert (
    phho["MOSAIC_FG"]
    == "NativePerennialForb"
)


# ============================================================================
# 13. BASSIA MANUAL ECOLOGICAL OVERRIDE
# ============================================================================

for code in [
    "BAPRV",
    "BAPRG",
]:

    x = one(code)

    assert (
        x["MOSAIC_FG"]
        == "ExoticForb"
    )

    assert (
        x["FG_resolution_source"]
        == "manual_ecological_functional_override"
    )


# ============================================================================
# 14. NEW STRUCTURAL-RULE SANITY CHECKS
#
# Lichenous -> Lichen
# Biennial forb -> BiennialForb
#
# These checks only require that examples exist if such traits are present.
# ============================================================================

lichen_rows = final.loc[
    final["USDA_growth_habit"]
    .astype("string")
    .eq("Lichenous")
    .fillna(False)
]

if len(lichen_rows) > 0:

    assert (
        lichen_rows[
            "MOSAIC_FG_structural"
        ]
        .eq("Lichen")
        .all()
    ), (
        "At least one pure Lichenous USDA taxon "
        "did not resolve to Lichen."
    )


biennial_forb_rows = final.loc[
    final["USDA_duration"]
    .astype("string")
    .eq("Biennial")
    .fillna(False)
    &
    final["USDA_growth_habit"]
    .astype("string")
    .isin(
        [
            "Forb/herb",
            "Forb/herb|Vine",
        ]
    )
]

#if len(biennial_forb_rows) > 0:
#
 #   assert (
  #      biennial_forb_rows[
   #         "MOSAIC_FG_structural"
    #    ]
     #   .eq("BiennialForb")
      #  .all()
    #), (
    #    "At least one pure Biennial forb "
    #    "did not resolve to BiennialForb."
    #)


# ============================================================================
# 15. RECORD-WEIGHTED SUMMARY TABLES
# ============================================================================

final["n_records_num"] = (
    pd.to_numeric(
        final["n_records"],
        errors="coerce",
    )
    .fillna(0)
)


# --------------------------------------------------------------------------
# Resolution-status summary
# --------------------------------------------------------------------------

resolution_summary = (
    final
    .groupby(
        "dictionary_resolution_status",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

resolution_summary["pct_all_records"] = (
    100
    * resolution_summary["n_records"]
    / final["n_records_num"].sum()
)


# --------------------------------------------------------------------------
# Plant-only summaries
# --------------------------------------------------------------------------

plant_records = (
    final.loc[
        plant_mask,
        "n_records_num",
    ]
    .sum()
)

structural_summary = (
    final.loc[
        plant_mask
    ]
    .groupby(
        "MOSAIC_FG_structural",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

structural_summary["pct_plant_records"] = (
    100
    * structural_summary["n_records"]
    / plant_records
)


refined_summary = (
    final.loc[
        plant_mask
    ]
    .groupby(
        "MOSAIC_FG",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

refined_summary["pct_plant_records"] = (
    100
    * refined_summary["n_records"]
    / plant_records
)


trait_summary = (
    final.loc[
        plant_mask
    ]
    .groupby(
        "USDA_trait_status",
        dropna=False,
    )
    .agg(
        n_codes=(
            "observed_code",
            "nunique",
        ),
        n_records=(
            "n_records_num",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "n_records",
        ascending=False,
    )
)

trait_summary["pct_plant_records"] = (
    100
    * trait_summary["n_records"]
    / plant_records
)


# ============================================================================
# 16. UNKNOWN / SPECIAL-PROTOCOL SUMMARY
# ============================================================================

unknown_special_summary = (
    final.loc[
        final[
            "dictionary_resolution_status"
        ].isin(
            [
                "UnknownPlant",
                "UnknownCode",
                "ProtocolPlantBase",
            ]
        )
    ]
    [
        [
            "observed_code",
            "canonical_code",
            "canonical_label",
            "code_class",
            "dictionary_resolution_status",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "hit_interpretation",
            "n_records_num",
        ]
    ]
    .sort_values(
        "n_records_num",
        ascending=False,
    )
)


# ============================================================================
# 17. DISPLAY QA SUMMARIES
# ============================================================================

print("=" * 90)
print("DICTIONARY RESOLUTION STATUS")
print("=" * 90)

display(
    resolution_summary
)


print("\n" + "=" * 90)
print("STRUCTURAL FUNCTIONAL GROUPS — PLANT ROWS ONLY")
print("=" * 90)

display(
    structural_summary
)


print("\n" + "=" * 90)
print("NATIVITY-REFINED FUNCTIONAL GROUPS — PLANT ROWS ONLY")
print("=" * 90)

display(
    refined_summary
)


print("\n" + "=" * 90)
print("USDA TRAIT COMPLETENESS — DIAGNOSTIC ONLY")
print("=" * 90)

display(
    trait_summary
)


print("\n" + "=" * 90)
print("UNKNOWN / SPECIAL-PROTOCOL TAIL")
print("=" * 90)

display(
    unknown_special_summary.head(100)
)


# ============================================================================
# 18. FINAL GATE
# ============================================================================

print("\n" + "=" * 90)
print("FINAL QA/QC: PASS")
print("=" * 90)

print(
    f"Dictionary codes:      {len(final):,}"
)

print(
    f"Represented LPI hits:  "
    f"{final['n_records_num'].sum():,.0f}"
)

print(
    f"UnknownPlant codes:    "
    f"{unknown_plant_mask.sum():,}"
)

print(
    f"UnknownCode codes:     "
    f"{unknown_code_mask.sum():,}"
)

print(
    "PlantBase records:     "
    f"{int(plant_base['n_records']):,}"
)

print(
    "\nAll observed codes are retained, "
    "all records are accounted for, "
    "and every hit has an explicit interpretation."
)

Oddball code retained: GUTcfSAR -> USDA_corrected_or_synonym_taxon / Woody
DICTIONARY RESOLUTION STATUS


,dictionary_resolution_status,n_codes,n_records,pct_all_records
1,NonPlant_classified,45,10270278,63.592655
5,USDA_exact_taxon,7160,5162304,31.964531
0,NoCanopy,1,442612,2.740614
2,NonUSDA_plant_functionally_resolved,3002,198192,1.227187
4,USDA_corrected_or_synonym_taxon,446,54735,0.338914
3,ProtocolPlantBase,1,17980,0.111331
7,UnknownPlant,141,3134,0.019405
6,UnknownCode,164,866,0.005362



STRUCTURAL FUNCTIONAL GROUPS — PLANT ROWS ONLY


,MOSAIC_FG_structural,n_codes,n_records,pct_plant_records
10,PerennialGrass,1492,2267084,41.840740
17,Woody,1373,769071,14.193784
1,AnnualGrass,312,688574,12.708151
11,Shrub,940,549385,10.139313
0,AnnualForb,2081,337610,6.230846
9,PerennialForb,3283,274126,5.059201
14,Tree,209,271072,5.002838
4,Forb,329,76238,1.407030
5,Grass,106,59283,1.094112
12,Subshrub,106,36722,0.677732



NATIVITY-REFINED FUNCTIONAL GROUPS — PLANT ROWS ONLY


...


USDA TRAIT COMPLETENESS — DIAGNOSTIC ONLY


,USDA_trait_status,n_codes,n_records,pct_plant_records
5,Traits_complete,7145,5191821,95.818960
4,No_USDA_taxon,3143,201326,3.715623
2,Missing_duration_nativity,231,12894,0.237968
3,Missing_nativity,154,11530,0.212795
1,Missing_duration_growth_habit_nativity,73,737,0.013602
0,Missing_duration,2,53,0.000978
6,NaN,1,4,0.000074



UNKNOWN / SPECIAL-PROTOCOL TAIL


,observed_code,canonical_code,canonical_label,code_class,dictionary_resolution_status,MOSAIC_FG_structural,MOSAIC_FG,hit_interpretation,n_records_num
111,Plant base,Plant base,Plant basal contact,PlantBase,ProtocolPlantBase,<NA>,<NA>,PlantBase,17980
963,BOETRII,BOETRII,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,476
1865,PINSCO,PINSCO,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,441
1155,PHYGOR,PHYGOR,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,365
948,PHYFEN,PHYFEN,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,290
1790,SIDTEN,SIDTEN,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,246
1392,SPOCFCRY,SPOCFCRY,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,220
1224,Unk,Unk,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,195
1794,UNK,UNK,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,86
2786,BOERHAF,BOERHAF,Unknown plant,Plant,UnknownPlant,UnknownPlant,UnknownPlant,UnknownPlant,75



FINAL QA/QC: PASS
Dictionary codes:      10,960
Represented LPI hits:  16,150,101
UnknownPlant codes:    141
UnknownCode codes:     164
PlantBase records:     17,980

All observed codes are retained, all records are accounted for, and every hit has an explicit interpretation.


## 22. Write the final master once

No earlier cell writes `LDC_LPI_species_dictionary_FINAL_MASTER.csv`. If the QA cell fails, this cell should not be run.


In [123]:
# ============================================================================
# FINAL WRITE
# ============================================================================

# Drop temporary analysis-only numeric helper before persistence.
final_to_write = final.drop(
    columns=["n_records_num"],
    errors="ignore",
).copy()

final_to_write.to_csv(
    FINAL_MASTER_FILE,
    index=False,
)

# Read-back verification.
check = pd.read_csv(
    FINAL_MASTER_FILE,
    dtype="string",
    low_memory=False,
)

assert len(check) == len(final_to_write)
assert check["observed_code"].nunique(dropna=False) == len(check)
assert check["observed_code"].eq("__NO_CANOPY__").sum() == 1
assert check["observed_code"].eq("NONE").sum() == 1
assert check["hit_interpretation"].notna().all()
assert check["canonical_code"].notna().all()
assert check["canonical_label"].notna().all()
assert (
    check.loc[
        check["observed_code"].eq("BRTE"),
        "MOSAIC_FG",
    ].iloc[0]
    == "EAG"
)

print("Final master written and verified:")
print(FINAL_MASTER_FILE)
print(f"Rows: {len(check):,}")


Final master written and verified:
C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv
Rows: 10,960


## 23. Optional unresolved-tail audit

This section is descriptive only. It does not modify the dictionary. Use it after a successful build to decide whether remaining unknown codes merit further manual resolution.


In [124]:
# ============================================================================
# OPTIONAL RECORD-WEIGHTED UNRESOLVED AUDIT
# ============================================================================

audit = final.copy()

unresolved_labels = {
    "TraitUnresolvedPlant",
    "UnknownPlant",
    "MixedGrowthHabit",
    "Grass",
    "Forb",
}

audit["is_broad_or_unresolved"] = (
    audit["code_class"].eq("Plant")
    & audit["MOSAIC_FG"].isin(unresolved_labels)
)

unresolved = (
    audit.loc[audit["is_broad_or_unresolved"]]
    .sort_values("n_records_num", ascending=False)
    .copy()
)

summary = (
    unresolved
    .groupby(
        ["MOSAIC_FG", "USDA_trait_status"],
        dropna=False,
    )
    .agg(
        n_codes=("observed_code", "nunique"),
        n_records=("n_records_num", "sum"),
    )
    .reset_index()
    .sort_values("n_records", ascending=False)
)

summary["pct_of_all_plant_records"] = (
    100 * summary["n_records"] / plant_records
)

display(summary)

display(
    unresolved[
        [
            "observed_code",
            "resolved_label",
            "USDA_accepted_symbol",
            "USDA_duration",
            "USDA_growth_habit",
            "USDA_native_status_L48",
            "MOSAIC_FG_structural",
            "MOSAIC_FG",
            "USDA_trait_status",
            "n_records_num",
            "n_plot_visits",
        ]
    ].head(100)
)


,MOSAIC_FG,USDA_trait_status,n_codes,n_records,pct_of_all_plant_records
5,Grass,Traits_complete,77,57562,1.062350
7,MixedGrowthHabit,Traits_complete,50,16453,0.303652
9,UnknownPlant,No_USDA_taxon,141,3134,0.057840
2,Grass,Missing_duration_nativity,26,1646,0.030378
8,TraitUnresolvedPlant,Missing_duration_growth_habit_nativity,73,737,0.013602
6,MixedGrowthHabit,Missing_nativity,1,117,0.002159
4,Grass,No_USDA_taxon,2,74,0.001366
0,Forb,Missing_nativity,2,7,0.000129
1,Forb,No_USDA_taxon,1,7,0.000129
3,Grass,Missing_nativity,1,1,0.000018


,observed_code,resolved_label,USDA_accepted_symbol,USDA_duration,USDA_growth_habit,USDA_native_status_L48,MOSAIC_FG_structural,MOSAIC_FG,USDA_trait_status,n_records_num,n_plot_visits
43,ARPU9,Aristida purpurea Nutt.,ARPU9,Annual|Perennial,Graminoid,N,Grass,Grass,Traits_complete,21232,2881
85,CAREX,Carex L.,CAREX,Annual|Perennial,Forb/herb|Graminoid,I,MixedGrowthHabit,MixedGrowthHabit,Traits_complete,12263,1498
327,DIGIT2,Digitaria Haller,DIGIT2,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,4792,265
377,LOPE,Lolium perenne L.,LOPE,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,4037,222
489,SPORO,Sporobolus R. Br.,SPORO,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,3128,160
534,BRDI3,Bromus diandrus Roth,BRDI3,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,2576,142
313,JUNCU,Juncus L.,JUNCU,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,2560,290
396,POA,Poa L.,POA,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,2426,209
268,ARIST,Aristida L.,ARIST,Annual|Perennial,Graminoid,N,Grass,Grass,Traits_complete,2372,350
919,LOLIU,Lolium L.,LOLIU,Annual|Perennial,Graminoid,I,Grass,Grass,Traits_complete,1367,68


# Appendix: intentionally excluded from the production path

The original notebook mixed production dictionary construction with exploratory debugging and downstream plot-visit extraction. Those tasks remain excluded.

Still excluded:

- recursive USDA JSON inspection/debugging;
- experimental parser variants;
- one-off malformed-cache surgery;
- stale `USDA_PLANTS_ecological_attributes_final.csv` use;
- repeated repair/rejoin of an already-written final master;
- repeated final QA cells;
- multiple final/intermediate dictionary writes;
- georeferenced LPI plot-visit extraction.

What **is** now included is a bounded, deterministic cache-maintenance stage:

**derive required accepted symbols → audit cache → query absent or unverified-incomplete profiles with the validated root parser → update one authoritative cache → continue**

This makes the species-dictionary build reproducible without reintroducing the exploratory recovery machinery. Plot-visit and composition generation remain downstream tasks.
